Modelo               | Tipo           | AUC-PR | F1 | KB | ms | Int8 | OTA

STA/LTA              | clássico       |       |    | NA |    | NA   | NA

Random Forest        | ML features    |       |    |    |    | talvez | sim

ExtraTrees           | ML features    |       |    |    |    | talvez | sim

Dense AE             | DL baseline    |       |    |    |    | sim  | sim

CNN 1D AE antiga     | DL AE          |       |    |    |    | sim  | sim

Tiny CNN AE          | TinyML AE      |       |    |    |    | sim  | sim

Tiny CNN classifier  | TinyML sup.    |       |    |    |    | sim  | sim

GRU AE               | DL seq. leve   |       |    |    |    | difícil | talvez

LSTM AE              | DL seq. pesado |       |    |    |    | difícil | talvez

In [ ]:
%pip install optuna tensorflow keras scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.4 MB/s eta 0:00:00


## Dataset

In [ ]:
# Cria o split por evento (v4) a partir do dataset v3 já processado.
#
# Pronto para Google Colab:
# - monta o Google Drive automaticamente, quando estiver no Colab;
# - usa caminhos em /content/drive/MyDrive/...;
# - valida se os arquivos necessários existem;
# - salva o dataset_v4_split_evento.npz, inventário CSV e JSON de resumo.
#
# Arquivos esperados dentro de DATA_DIR:
# - inventario_v3.csv
# - dataset_v3_split_temporal.npz
#
# Saídas geradas dentro de DATA_DIR:
# - dataset_v4_split_evento.npz
# - inventario_v4_split_evento.csv
# - dataset_v4_split_evento_info.json


import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuração para Colab / Drive
# -----------------------------------------------------------------------------

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except Exception:
    # Fora do Colab, essa etapa é ignorada.
    pass

DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed",
    )
).expanduser()

SEED = 42


# -----------------------------------------------------------------------------
# Funções auxiliares
# -----------------------------------------------------------------------------

def validar_arquivos_entrada(split_base: str) -> tuple[Path, Path]:
    """Valida a existência dos arquivos necessários para reconstruir o dataset."""

    inventario_path = DATA_DIR / "inventario_v3.csv"
    dataset_path = DATA_DIR / f"dataset_v3_split_{split_base}.npz"

    faltando = []
    if not inventario_path.exists():
        faltando.append(inventario_path.name)
    if not dataset_path.exists():
        faltando.append(dataset_path.name)

    if faltando:
        raise FileNotFoundError(
            "Arquivos necessários não encontrados.\n"
            f"Pasta procurada: {DATA_DIR}\n"
            f"Arquivos faltando: {', '.join(faltando)}\n\n"
            "No Google Drive, deixe os arquivos nesta estrutura:\n"
            "Meu Drive/AlvaroSampaio/TCC/TCC_data/processed/\n"
            "  ├── inventario_v3.csv\n"
            f"  └── dataset_v3_split_{split_base}.npz\n\n"
            "Se quiser usar outro caminho, defina antes de rodar:\n"
            "import os\n"
            "os.environ['TCC_PROCESSED_DIR'] = '/content/drive/MyDrive/seu/caminho/processed'"
        )

    return inventario_path, dataset_path


def reconstruir_dataset_total(split_base: str = "temporal"):
    """Reconstrói X_all e y_all na ordem original do inventário v3."""

    inventario_path, dataset_path = validar_arquivos_entrada(split_base)

    print("=" * 80)
    print("RECONSTRUINDO DATASET TOTAL")
    print("=" * 80)
    print(f"DATA_DIR:   {DATA_DIR}")
    print(f"Inventário: {inventario_path}")
    print(f"Dataset:    {dataset_path}")

    df = pd.read_csv(inventario_path)
    data = np.load(dataset_path)

    split_col = f"split_{split_base}"

    colunas_obrigatorias = {"tipo", "evid", split_col}
    colunas_faltando = colunas_obrigatorias - set(df.columns)
    if colunas_faltando:
        raise ValueError(
            "O inventario_v3.csv não possui as colunas necessárias: "
            f"{sorted(colunas_faltando)}"
        )

    chaves_obrigatorias = {
        "X_train", "y_train",
        "X_val", "y_val",
        "X_test", "y_test",
    }
    chaves_faltando = chaves_obrigatorias - set(data.files)
    if chaves_faltando:
        raise ValueError(
            "O arquivo NPZ não possui as chaves necessárias: "
            f"{sorted(chaves_faltando)}"
        )

    tipos_validos = {"normal", "anomalo"}
    tipos_encontrados = set(df["tipo"].dropna().unique())
    tipos_invalidos = tipos_encontrados - tipos_validos
    if tipos_invalidos:
        raise ValueError(
            "A coluna 'tipo' possui valores inesperados: "
            f"{sorted(tipos_invalidos)}. Esperado apenas: {sorted(tipos_validos)}"
        )

    n_total = len(df)
    sample_shape = data["X_train"].shape[1:]
    dtype_x = data["X_train"].dtype
    dtype_y = data["y_train"].dtype

    X_all = np.empty((n_total, *sample_shape), dtype=dtype_x)
    y_all = np.empty(n_total, dtype=dtype_y)

    for parte in ["train", "val", "test"]:
        idx = df.index[df[split_col] == parte].to_numpy()

        X_parte = data[f"X_{parte}"]
        y_parte = data[f"y_{parte}"]

        if len(idx) != len(X_parte):
            raise ValueError(
                f"Tamanho inconsistente em {parte}: "
                f"inventario={len(idx)} | npz={len(X_parte)}"
            )

        X_all[idx] = X_parte
        y_all[idx] = y_parte

    y_inventario = (df["tipo"] == "anomalo").astype(dtype_y).to_numpy()

    if not np.array_equal(y_all, y_inventario):
        raise ValueError("y_all reconstruído não bate com inventario_v3.csv")

    print("OK: dataset total reconstruído.")
    print(f"X_all: {X_all.shape}")
    print(f"y_all: {y_all.shape}")
    print()

    return X_all, y_all, df


def dividir_ids(ids, rng, train_ratio: float = 0.70, val_ratio: float = 0.15):
    """Divide IDs únicos em treino, validação e teste."""

    ids = np.array(sorted(ids))
    rng.shuffle(ids)

    n = len(ids)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    ids_train = set(ids[:n_train])
    ids_val = set(ids[n_train:n_train + n_val])
    ids_test = set(ids[n_train + n_val:])

    return ids_train, ids_val, ids_test


def criar_split_evento(df: pd.DataFrame):
    """Cria split garantindo que o mesmo evento anômalo não vaze entre conjuntos."""

    rng = np.random.default_rng(SEED)

    splits = np.full(len(df), fill_value="", dtype=object)

    mask_anomalo = df["tipo"] == "anomalo"
    mask_normal = df["tipo"] == "normal"

    # 1. Eventos anômalos são divididos por evid.
    # Assim, o mesmo terremoto não aparece em mais de um split.
    event_ids = df.loc[mask_anomalo, "evid"].dropna().unique()
    ev_train, ev_val, ev_test = dividir_ids(event_ids, rng)

    for idx, row in df.loc[mask_anomalo].iterrows():
        evid = row["evid"]

        if evid in ev_train:
            splits[idx] = "train"
        elif evid in ev_val:
            splits[idx] = "val"
        elif evid in ev_test:
            splits[idx] = "test"
        else:
            raise ValueError(f"Evento anômalo sem split definido: evid={evid}")

    # 2. Normais são divididos aleatoriamente.
    # Como janelas normais têm evid único, não existe vazamento por evento.
    normal_idx = df.index[mask_normal].to_numpy()
    rng.shuffle(normal_idx)

    n_norm = len(normal_idx)
    n_train = int(n_norm * 0.70)
    n_val = int(n_norm * 0.15)

    idx_train = normal_idx[:n_train]
    idx_val = normal_idx[n_train:n_train + n_val]
    idx_test = normal_idx[n_train + n_val:]

    splits[idx_train] = "train"
    splits[idx_val] = "val"
    splits[idx_test] = "test"

    if np.any(splits == ""):
        linhas_sem_split = int(np.sum(splits == ""))
        raise ValueError(f"Existem {linhas_sem_split} linhas sem split definido.")

    return splits


def salvar_split(nome: str, X_all, y_all, df: pd.DataFrame, splits):
    """Salva o novo split em NPZ, CSV e JSON."""

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    out_npz = DATA_DIR / f"dataset_v4_split_{nome}.npz"
    out_csv = DATA_DIR / f"inventario_v4_split_{nome}.csv"
    out_json = DATA_DIR / f"dataset_v4_split_{nome}_info.json"

    payload = {}

    for parte in ["train", "val", "test"]:
        mask = splits == parte
        payload[f"X_{parte}"] = X_all[mask]
        payload[f"y_{parte}"] = y_all[mask]

    np.savez_compressed(out_npz, **payload)

    df_out = df.copy()
    df_out[f"split_{nome}"] = splits
    df_out.to_csv(out_csv, index=False)

    counts = {}

    for parte in ["train", "val", "test"]:
        y = payload[f"y_{parte}"]

        normal = int((y == 0).sum())
        anomalo = int((y == 1).sum())
        total = int(len(y))

        counts[parte] = {
            "total": total,
            "normal": normal,
            "anomalo": anomalo,
            "baseline_auc_pr": anomalo / total if total else 0.0,
        }

    eventos = df_out[df_out["tipo"] == "anomalo"]
    vazamento = eventos.groupby("evid")[f"split_{nome}"].nunique()
    n_eventos_vazando = int((vazamento > 1).sum())

    info = {
        "nome": nome,
        "seed": SEED,
        "data_dir": str(DATA_DIR),
        "arquivos": {
            "dataset": str(out_npz),
            "inventario": str(out_csv),
            "info": str(out_json),
        },
        "counts": counts,
        "n_eventos_vazando": n_eventos_vazando,
    }

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    print()
    print("=" * 80)
    print(f"SPLIT V4 SALVO: {nome}")
    print("=" * 80)
    print(f"NPZ:  {out_npz}")
    print(f"CSV:  {out_csv}")
    print(f"JSON: {out_json}")

    print()
    print("Distribuição:")
    print(pd.crosstab(df_out[f"split_{nome}"], df_out["tipo"]))

    print()
    print("Baseline AUC-PR por parte:")
    for parte, c in counts.items():
        print(f"{parte:5s}: {c['baseline_auc_pr']:.4f}")

    print()
    if n_eventos_vazando == 0:
        print("OK: nenhum evid anômalo aparece em mais de um split.")
    else:
        print(f"ALERTA: {n_eventos_vazando} eventos aparecem em mais de um split.")


def main():
    print("=" * 80)
    print("CRIANDO DATASET V4 COM SPLIT POR EVENTO")
    print("=" * 80)
    print(f"Pasta de dados processados: {DATA_DIR}")
    print()

    X_all, y_all, df = reconstruir_dataset_total(split_base="temporal")
    splits_evento = criar_split_evento(df)

    salvar_split(
        nome="evento",
        X_all=X_all,
        y_all=y_all,
        df=df,
        splits=splits_evento,
    )


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CRIANDO DATASET V4 COM SPLIT POR EVENTO
Pasta de dados processados: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed

RECONSTRUINDO DATASET TOTAL
DATA_DIR:   /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed
Inventário: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/inventario_v3.csv
Dataset:    /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v3_split_temporal.npz
OK: dataset total reconstruído.
X_all: (172060, 800)
y_all: (172060,)


SPLIT V4 SALVO: evento
NPZ:  /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
CSV:  /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/inventario_v4_split_evento.csv
JSON: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento_info.json

Distribuição:
tipo          anomalo  normal
split_evento       

## Aprendizado ensemble

In [ ]:
import csv
import json
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
from scipy.signal import welch
from scipy.stats import kurtosis, skew
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import joblib
except Exception:
    joblib = None


# -------------------------------------------------------------------------
# CONFIGURAÇÃO PARA GOOGLE COLAB / DRIVE
# -------------------------------------------------------------------------

# Monta o Google Drive automaticamente quando estiver no Colab.
# Se estiver rodando fora do Colab, essa parte é ignorada.
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except Exception:
    pass

# Caminho base do projeto no Google Drive.
# Altere aqui se sua pasta tiver outro nome.
BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

# Pasta onde está o arquivo .npz processado.
DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()

# Pasta onde os resultados serão salvos.
RESULTS_DIR = Path(
    os.environ.get(
        "TCC_RESULTS_DIR",
        str(BASE_DIR / "results" / "rf"),
    )
).expanduser()

# Nome do dataset usado por este baseline.
DATASET_FILENAME = os.environ.get("TCC_DATASET_FILENAME", "dataset_v4_split_evento.npz")
DATASET_PATH = DATA_DIR / DATASET_FILENAME

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SR = 40.0
SEED = 42

FEATURE_NAMES = [
    "media",
    "std",
    "rms",
    "energia",
    "pico_abs",
    "peak_to_peak",
    "kurtosis",
    "skewness",
    "zero_crossing_rate",
    "energia_0_3hz",
    "energia_3_8hz",
    "energia_8_15hz",
    "entropia_espectral",
]


# -------------------------------------------------------------------------
# MÉTRICAS E THRESHOLD
# -------------------------------------------------------------------------

def escolher_threshold_validacao(y_true, scores):
    """Escolhe o threshold que maximiza F1 no conjunto de validação."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=np.float64)

    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    if len(thresholds) == 0:
        return {
            "threshold": 0.5,
            "f1": 0.0,
            "precision": 0.0,
            "recall": 0.0,
        }

    # O último ponto de precision/recall não possui threshold correspondente.
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1))

    return {
        "threshold": float(thresholds[best_idx]),
        "f1": float(f1[best_idx]),
        "precision": float(precision[best_idx]),
        "recall": float(recall[best_idx]),
    }


def avaliar_com_threshold(y_true, scores, threshold):
    """Avalia o modelo usando um threshold fixo."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=np.float64)
    y_pred = (scores >= threshold).astype(int)

    labels = [0, 1]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=labels).ravel()

    resultado = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    # ROC-AUC só existe se houver as duas classes no y_true.
    if len(np.unique(y_true)) == 2:
        resultado["roc_auc"] = float(roc_auc_score(y_true, scores))
    else:
        resultado["roc_auc"] = None

    return resultado


# -------------------------------------------------------------------------
# EXTRAÇÃO DE FEATURES
# -------------------------------------------------------------------------

def extrair_feature_janela(x):
    """Extrai features de uma janela de sinal sísmico."""
    x = np.asarray(x, dtype=np.float32).reshape(-1)

    media = np.mean(x)
    std = np.std(x)
    rms = np.sqrt(np.mean(x**2))
    energia = np.sum(x**2)
    pico_abs = np.max(np.abs(x))
    peak_to_peak = np.ptp(x)
    kurt = kurtosis(x, fisher=True, bias=False)
    sk = skew(x, bias=False)

    sinais = np.sign(x)
    zero_crossing_rate = np.sum(sinais[:-1] != sinais[1:])

    freqs, psd = welch(
        x,
        fs=SR,
        nperseg=min(256, len(x)),
    )

    energia_total = np.sum(psd) + 1e-12
    energia_0_3 = np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)]) / energia_total
    energia_3_8 = np.sum(psd[(freqs >= 3.0) & (freqs < 8.0)]) / energia_total
    energia_8_15 = np.sum(psd[(freqs >= 8.0) & (freqs < 15.0)]) / energia_total

    p = psd / energia_total
    entropia = -np.sum(p * np.log(p + 1e-12))

    return np.array(
        [
            media,
            std,
            rms,
            energia,
            pico_abs,
            peak_to_peak,
            kurt,
            sk,
            zero_crossing_rate,
            energia_0_3,
            energia_3_8,
            energia_8_15,
            entropia,
        ],
        dtype=np.float32,
    )


def extrair_features(X, nome):
    print(f"Extraindo features: {nome} | X = {X.shape}")

    feats = np.vstack([extrair_feature_janela(x) for x in X])
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Features extraídas: {feats.shape}")
    return feats


# -------------------------------------------------------------------------
# CARREGAMENTO DO DATASET
# -------------------------------------------------------------------------

def carregar_dataset():
    print("=" * 80)
    print("BASELINE FEATURES + RANDOM FOREST")
    print("=" * 80)
    print("Diretórios configurados:")
    print(f"  BASE_DIR    : {BASE_DIR}")
    print(f"  DATA_DIR    : {DATA_DIR}")
    print(f"  RESULTS_DIR : {RESULTS_DIR}")
    print(f"  DATASET     : {DATASET_PATH}")

    if not DATASET_PATH.exists():
        encontrados = []
        if DATA_DIR.exists():
            encontrados = sorted(p.name for p in DATA_DIR.glob("*.npz"))

        raise FileNotFoundError(
            "Arquivo de dataset não encontrado.\n"
            f"Procurado em: {DATASET_PATH}\n\n"
            "Coloque o arquivo .npz nesse caminho ou altere TCC_PROCESSED_DIR/TCC_DATASET_FILENAME.\n"
            f"Arquivos .npz encontrados em DATA_DIR: {encontrados}"
        )

    data = np.load(DATASET_PATH, allow_pickle=False)
    required_keys = {"X_train", "y_train", "X_val", "y_val", "X_test", "y_test"}
    missing = sorted(required_keys - set(data.files))

    if missing:
        raise KeyError(
            "O arquivo .npz não possui todas as chaves necessárias.\n"
            f"Chaves faltando: {missing}\n"
            f"Chaves encontradas: {list(data.files)}"
        )

    X_train = data["X_train"]
    y_train = data["y_train"].astype(int)
    X_val = data["X_val"]
    y_val = data["y_val"].astype(int)
    X_test = data["X_test"]
    y_test = data["y_test"].astype(int)

    for nome, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
        n_evento = int(np.sum(y == 1))
        n_total = int(len(y))
        perc = n_evento / n_total if n_total else 0.0
        print(f"{nome}: {n_evento} eventos anômalos de {n_total} amostras ({perc:.2%})")

    return X_train, y_train, X_val, y_val, X_test, y_test


# -------------------------------------------------------------------------
# TREINO E AVALIAÇÃO
# -------------------------------------------------------------------------

def main():
    X_train, y_train, X_val, y_val, X_test, y_test = carregar_dataset()

    X_train_f = extrair_features(X_train, "train")
    X_val_f = extrair_features(X_val, "val")
    X_test_f = extrair_features(X_test, "test")

    modelo = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "rf",
                RandomForestClassifier(
                    n_estimators=400,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced_subsample",
                    random_state=SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    print("Treinando modelo...")
    modelo.fit(X_train_f, y_train)

    scores_val = modelo.predict_proba(X_val_f)[:, 1]
    scores_test = modelo.predict_proba(X_test_f)[:, 1]

    escolha = escolher_threshold_validacao(y_val, scores_val)
    threshold = escolha["threshold"]
    print(f"Threshold escolhido na validação: {threshold:.6f}")

    res_val = avaliar_com_threshold(y_val, scores_val, threshold)
    res_test = avaliar_com_threshold(y_test, scores_test, threshold)

    rf = modelo.named_steps["rf"]
    importancias = rf.feature_importances_
    ranking = np.argsort(importancias)[::-1]

    feature_importances = [
        {"feature": FEATURE_NAMES[i], "importance": float(importancias[i])}
        for i in ranking
    ]

    resultado = {
        "modelo": "features_random_forest",
        "dataset": str(DATASET_PATH),
        "threshold_escolhido_na_validacao": float(threshold),
        "escolha_threshold": escolha,
        "validacao": res_val,
        "teste": res_test,
        "features": FEATURE_NAMES,
        "feature_importances": feature_importances,
    }

    out_json = RESULTS_DIR / "resultado_rf.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False)

    out_csv = RESULTS_DIR / "feature_importances_rf.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["feature", "importance"])
        writer.writeheader()
        writer.writerows(feature_importances)

    if joblib is not None:
        out_model = RESULTS_DIR / "modelo_rf.joblib"
        joblib.dump(modelo, out_model)
    else:
        out_model = None

    print()
    print("=" * 80)
    print("RESULTADO VALIDAÇÃO")
    print("=" * 80)
    print(json.dumps(res_val, indent=2, ensure_ascii=False))

    print()
    print("=" * 80)
    print("RESULTADO TESTE")
    print("=" * 80)
    print(json.dumps(res_test, indent=2, ensure_ascii=False))

    print()
    print("=" * 80)
    print("IMPORTÂNCIA DAS FEATURES")
    print("=" * 80)
    for item in feature_importances:
        print(f"{item['feature']:22s} {item['importance']:.4f}")

    print()
    print("Arquivos salvos:")
    print(f"  JSON métricas      : {out_json}")
    print(f"  CSV importâncias   : {out_csv}")
    if out_model is not None:
        print(f"  Modelo treinado    : {out_model}")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASELINE FEATURES + RANDOM FOREST
Diretórios configurados:
  BASE_DIR    : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data
  DATA_DIR    : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed
  RESULTS_DIR : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/results/rf
  DATASET     : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
train: 15429 eventos anômalos de 120429 amostras (12.81%)
val: 3302 eventos anômalos de 25802 amostras (12.80%)
test: 3329 eventos anômalos de 25829 amostras (12.89%)
Extraindo features: train | X = (120429, 800)
Features extraídas: (120429, 13)
Extraindo features: val | X = (25802, 800)
Features extraídas: (25802, 13)
Extraindo features: test | X = (25829, 800)
Features extraídas: (25829, 13)
Treinando modelo...
Threshold escolhido na validação: 0.373326

RESULTADO VALIDAÇÃO
{
  "thr

In [ ]:
import csv
import json
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
from scipy.signal import welch
from scipy.stats import kurtosis, skew
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import joblib
except Exception:
    joblib = None


# -------------------------------------------------------------------------
# CONFIGURAÇÃO PARA GOOGLE COLAB / DRIVE
# -------------------------------------------------------------------------

# Monta o Google Drive automaticamente quando estiver no Colab.
# Se estiver rodando fora do Colab, essa parte é ignorada.
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except Exception:
    pass

# Caminho base do projeto no Google Drive.
# Altere aqui se sua pasta tiver outro nome.
BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

# Pasta onde está o arquivo .npz processado.
DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()

# Pasta onde os resultados serão salvos.
RESULTS_DIR = Path(
    os.environ.get(
        "TCC_RESULTS_DIR",
        str(BASE_DIR / "results" / "et"),
    )
).expanduser()

# Nome do dataset usado por este baseline.
DATASET_FILENAME = os.environ.get("TCC_DATASET_FILENAME", "dataset_v4_split_evento.npz")
DATASET_PATH = DATA_DIR / DATASET_FILENAME

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SR = 40.0
SEED = 42

FEATURE_NAMES = [
    "media",
    "std",
    "rms",
    "energia",
    "pico_abs",
    "peak_to_peak",
    "kurtosis",
    "skewness",
    "zero_crossing_rate",
    "energia_0_3hz",
    "energia_3_8hz",
    "energia_8_15hz",
    "entropia_espectral",
]


# -------------------------------------------------------------------------
# MÉTRICAS E THRESHOLD
# -------------------------------------------------------------------------

def escolher_threshold_validacao(y_true, scores):
    """Escolhe o threshold que maximiza F1 no conjunto de validação."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=np.float64)

    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    if len(thresholds) == 0:
        return {
            "threshold": 0.5,
            "f1": 0.0,
            "precision": 0.0,
            "recall": 0.0,
        }

    # O último ponto de precision/recall não possui threshold correspondente.
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
    best_idx = int(np.nanargmax(f1))

    return {
        "threshold": float(thresholds[best_idx]),
        "f1": float(f1[best_idx]),
        "precision": float(precision[best_idx]),
        "recall": float(recall[best_idx]),
    }


def avaliar_com_threshold(y_true, scores, threshold):
    """Avalia o modelo usando um threshold fixo."""
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=np.float64)
    y_pred = (scores >= threshold).astype(int)

    labels = [0, 1]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=labels).ravel()

    resultado = {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "average_precision": float(average_precision_score(y_true, scores)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    # ROC-AUC só existe se houver as duas classes no y_true.
    if len(np.unique(y_true)) == 2:
        resultado["roc_auc"] = float(roc_auc_score(y_true, scores))
    else:
        resultado["roc_auc"] = None

    return resultado


# -------------------------------------------------------------------------
# EXTRAÇÃO DE FEATURES
# -------------------------------------------------------------------------

def extrair_feature_janela(x):
    """Extrai features de uma janela de sinal sísmico."""
    x = np.asarray(x, dtype=np.float32).reshape(-1)

    media = np.mean(x)
    std = np.std(x)
    rms = np.sqrt(np.mean(x**2))
    energia = np.sum(x**2)
    pico_abs = np.max(np.abs(x))
    peak_to_peak = np.ptp(x)
    kurt = kurtosis(x, fisher=True, bias=False)
    sk = skew(x, bias=False)

    sinais = np.sign(x)
    zero_crossing_rate = np.sum(sinais[:-1] != sinais[1:])

    freqs, psd = welch(
        x,
        fs=SR,
        nperseg=min(256, len(x)),
    )

    energia_total = np.sum(psd) + 1e-12
    energia_0_3 = np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)]) / energia_total
    energia_3_8 = np.sum(psd[(freqs >= 3.0) & (freqs < 8.0)]) / energia_total
    energia_8_15 = np.sum(psd[(freqs >= 8.0) & (freqs < 15.0)]) / energia_total

    p = psd / energia_total
    entropia = -np.sum(p * np.log(p + 1e-12))

    return np.array(
        [
            media,
            std,
            rms,
            energia,
            pico_abs,
            peak_to_peak,
            kurt,
            sk,
            zero_crossing_rate,
            energia_0_3,
            energia_3_8,
            energia_8_15,
            entropia,
        ],
        dtype=np.float32,
    )


def extrair_features(X, nome):
    print(f"Extraindo features: {nome} | X = {X.shape}")

    feats = np.vstack([extrair_feature_janela(x) for x in X])
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Features extraídas: {feats.shape}")
    return feats


# -------------------------------------------------------------------------
# CARREGAMENTO DO DATASET
# -------------------------------------------------------------------------

def carregar_dataset():
    print("=" * 80)
    print("BASELINE FEATURES + Extra tree")
    print("=" * 80)
    print("Diretórios configurados:")
    print(f"  BASE_DIR    : {BASE_DIR}")
    print(f"  DATA_DIR    : {DATA_DIR}")
    print(f"  RESULTS_DIR : {RESULTS_DIR}")
    print(f"  DATASET     : {DATASET_PATH}")

    if not DATASET_PATH.exists():
        encontrados = []
        if DATA_DIR.exists():
            encontrados = sorted(p.name for p in DATA_DIR.glob("*.npz"))

        raise FileNotFoundError(
            "Arquivo de dataset não encontrado.\n"
            f"Procurado em: {DATASET_PATH}\n\n"
            "Coloque o arquivo .npz nesse caminho ou altere TCC_PROCESSED_DIR/TCC_DATASET_FILENAME.\n"
            f"Arquivos .npz encontrados em DATA_DIR: {encontrados}"
        )

    data = np.load(DATASET_PATH, allow_pickle=False)
    required_keys = {"X_train", "y_train", "X_val", "y_val", "X_test", "y_test"}
    missing = sorted(required_keys - set(data.files))

    if missing:
        raise KeyError(
            "O arquivo .npz não possui todas as chaves necessárias.\n"
            f"Chaves faltando: {missing}\n"
            f"Chaves encontradas: {list(data.files)}"
        )

    X_train = data["X_train"]
    y_train = data["y_train"].astype(int)
    X_val = data["X_val"]
    y_val = data["y_val"].astype(int)
    X_test = data["X_test"]
    y_test = data["y_test"].astype(int)

    for nome, y in [("train", y_train), ("val", y_val), ("test", y_test)]:
        n_evento = int(np.sum(y == 1))
        n_total = int(len(y))
        perc = n_evento / n_total if n_total else 0.0
        print(f"{nome}: {n_evento} eventos anômalos de {n_total} amostras ({perc:.2%})")

    return X_train, y_train, X_val, y_val, X_test, y_test


# -------------------------------------------------------------------------
# TREINO E AVALIAÇÃO
# -------------------------------------------------------------------------

def main():
    X_train, y_train, X_val, y_val, X_test, y_test = carregar_dataset()

    X_train_f = extrair_features(X_train, "train")
    X_val_f = extrair_features(X_val, "val")
    X_test_f = extrair_features(X_test, "test")

    modelo = Pipeline([
    ("scaler", StandardScaler()),
    ("et", ExtraTreesClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )),
])


    print("Treinando modelo...")
    modelo.fit(X_train_f, y_train)

    scores_val = modelo.predict_proba(X_val_f)[:, 1]
    scores_test = modelo.predict_proba(X_test_f)[:, 1]

    escolha = escolher_threshold_validacao(y_val, scores_val)
    threshold = escolha["threshold"]
    print(f"Threshold escolhido na validação: {threshold:.6f}")

    res_val = avaliar_com_threshold(y_val, scores_val, threshold)
    res_test = avaliar_com_threshold(y_test, scores_test, threshold)

    et = modelo.named_steps["et"]
    importancias = et.feature_importances_
    ranking = np.argsort(importancias)[::-1]

    feature_importances = [
        {"feature": FEATURE_NAMES[i], "importance": float(importancias[i])}
        for i in ranking
    ]

    resultado = {
        "modelo": "features_extra_tree",
        "dataset": str(DATASET_PATH),
        "threshold_escolhido_na_validacao": float(threshold),
        "escolha_threshold": escolha,
        "validacao": res_val,
        "teste": res_test,
        "features": FEATURE_NAMES,
        "feature_importances": feature_importances,
    }

    out_json = RESULTS_DIR / "resultado_et.json"
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(resultado, f, indent=2, ensure_ascii=False)

    out_csv = RESULTS_DIR / "feature_importances_et.csv"
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["feature", "importance"])
        writer.writeheader()
        writer.writerows(feature_importances)

    if joblib is not None:
        out_model = RESULTS_DIR / "modelo_et.joblib"
        joblib.dump(modelo, out_model)
    else:
        out_model = None

    print()
    print("=" * 80)
    print("RESULTADO VALIDAÇÃO")
    print("=" * 80)
    print(json.dumps(res_val, indent=2, ensure_ascii=False))

    print()
    print("=" * 80)
    print("RESULTADO TESTE")
    print("=" * 80)
    print(json.dumps(res_test, indent=2, ensure_ascii=False))

    print()
    print("=" * 80)
    print("IMPORTÂNCIA DAS FEATURES")
    print("=" * 80)
    for item in feature_importances:
        print(f"{item['feature']:22s} {item['importance']:.4f}")

    print()
    print("Arquivos salvos:")
    print(f"  JSON métricas      : {out_json}")
    print(f"  CSV importâncias   : {out_csv}")
    if out_model is not None:
        print(f"  Modelo treinado    : {out_model}")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASELINE FEATURES + Extra tree
Diretórios configurados:
  BASE_DIR    : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data
  DATA_DIR    : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed
  RESULTS_DIR : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/results/et
  DATASET     : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
train: 15429 eventos anômalos de 120429 amostras (12.81%)
val: 3302 eventos anômalos de 25802 amostras (12.80%)
test: 3329 eventos anômalos de 25829 amostras (12.89%)
Extraindo features: train | X = (120429, 800)
Features extraídas: (120429, 13)
Extraindo features: val | X = (25802, 800)
Features extraídas: (25802, 13)
Extraindo features: test | X = (25829, 800)
Features extraídas: (25829, 13)
Treinando modelo...
Threshold escolhido na validação: 0.482113

RESULTADO VALIDAÇÃO
{
  "thresh

## Carregando o conjunto de dados

In [ ]:
import os
from pathlib import Path
SEED = 42
SR = 40.0
WINDOW_NPTS = 800
WINDOW_SECONDS = 20.0
STEP_SECONDS = 10.0

BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

#pasta onde ficam os datasets processados
DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()
RESULTS_DIR = BASE_DIR / "resultados"
DATA_DIR.mkdir(parents = True, exist_ok = True)
DATASET_V4 = DATA_DIR/'dataset_v4_split_evento.npz'
INVENTORY_V4 = DATA_DIR / 'inventario_v4_split_evento.csv'

def model_dir(name: str) -> Path:
  path = RESULTS_DIR / name
  path.mkdir(parents=True, exist_ok=True)
  return path

print("Diretórios configurados:")
print(f"BASE_DIR    : {BASE_DIR}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"RESULTS_DIR : {RESULTS_DIR}")
print(f"DATASET_V4  : {DATASET_V4}")
print(f"INVENTORY_V4: {INVENTORY_V4}")

if not DATASET_V4.exists():
    print(f"AVISO: dataset não encontrado em: {DATASET_V4}")

if not INVENTORY_V4.exists():
    print(f"AVISO: inventário não encontrado em: {INVENTORY_V4}")

Diretórios configurados:
BASE_DIR    : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data
DATA_DIR    : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed
RESULTS_DIR : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados
DATASET_V4  : /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
INVENTORY_V4: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/inventario_v4_split_evento.csv


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np


WINDOW_NPTS = 800

DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed"
    )
).expanduser()

DATASET_V4 = DATA_DIR / "dataset_v4_split_evento.npz"


def load_dataset(path=DATASET_V4) -> dict[str, np.ndarray]:
    path = Path(path).expanduser()

    if not path.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em:\n{path}\n\n"
            "Verifique se o arquivo dataset_v4_split_evento.npz está na pasta processed do Google Drive."
        )

    data = np.load(path)

    required_keys = [
        "X_train", "y_train",
        "X_val", "y_val",
        "X_test", "y_test",
    ]

    missing = [key for key in required_keys if key not in data.files]

    if missing:
        raise KeyError(
            f"O arquivo .npz não possui as chaves esperadas: {missing}\n"
            f"Chaves encontradas: {data.files}"
        )

    return {
        "X_train": data["X_train"].astype(np.float32),
        "y_train": data["y_train"].astype(np.int8),
        "X_val": data["X_val"].astype(np.float32),
        "y_val": data["y_val"].astype(np.int8),
        "X_test": data["X_test"].astype(np.float32),
        "y_test": data["y_test"].astype(np.int8),
    }


def class_summary(y: np.ndarray) -> dict:
    y = np.asarray(y)

    total = int(len(y))
    normal = int((y == 0).sum())
    event = int((y == 1).sum())

    return {
        "total": total,
        "normal": normal,
        "event": event,
        "baseline_auc_pr": event / total if total else 0.0,
    }


def split_normal_event(X: np.ndarray, y: np.ndarray):
    X = np.asarray(X)
    y = np.asarray(y)

    return X[y == 0], X[y == 1]


def to_cnn(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 3 and X.shape[-1] == 1:
        return X.astype(np.float32)

    if X.shape[-1] != WINDOW_NPTS:
        X = X.reshape(-1, WINDOW_NPTS)

    return X.reshape(-1, WINDOW_NPTS, 1).astype(np.float32)


def to_seq(X: np.ndarray, timesteps: int, features: int) -> np.ndarray:
    X = np.asarray(X, dtype=np.float32)

    expected = timesteps * features

    if X.shape[-1] != expected:
        X = X.reshape(-1, expected)

    return X.reshape(-1, timesteps, features).astype(np.float32)


def make_class_weights(y: np.ndarray) -> dict[int, float]:
    y = np.asarray(y)

    n = len(y)
    n0 = max(int((y == 0).sum()), 1)
    n1 = max(int((y == 1).sum()), 1)

    return {
        0: n / (2 * n0),
        1: n / (2 * n1),
    }


if __name__ == "__main__":
    dataset = load_dataset()

    print("Dataset carregado com sucesso.")
    print("X_train:", dataset["X_train"].shape)
    print("y_train:", dataset["y_train"].shape)
    print("X_val:", dataset["X_val"].shape)
    print("y_val:", dataset["y_val"].shape)
    print("X_test:", dataset["X_test"].shape)
    print("y_test:", dataset["y_test"].shape)

    print()
    print("Resumo treino:")
    print(class_summary(dataset["y_train"]))

    print()
    print("Resumo validação:")
    print(class_summary(dataset["y_val"]))

    print()
    print("Resumo teste:")
    print(class_summary(dataset["y_test"]))


Dataset carregado com sucesso.
X_train: (120429, 800)
y_train: (120429,)
X_val: (25802, 800)
y_val: (25802,)
X_test: (25829, 800)
y_test: (25829,)

Resumo treino:
{'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}

Resumo validação:
{'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}

Resumo teste:
{'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import numpy as np
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score,
                             roc_auc_score)

def choose_threshold_max_f1(y_val: np.ndarray, scores_val: np.ndarray, threshold: float) ->dict :
  y_pred = (scores_val >= threshold).astype(np.int8)
  tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels = [0,1]).ravel()
  fpr = fp/max(tn+fp,1) # fpr = false positive rate
  # STEP_SECONDS is defined in cell Lo7L2yi9xMsb
  windows_per_hour = 3600 / STEP_SECONDS
  return {
        "auc_pr": float(average_precision_score(y_val, scores_val)),
        "auc_roc": float(roc_auc_score(y_val, scores_val)),
        "threshold": float(threshold),
        "precision": float(precision_score(y_val, y_pred, zero_division=0)),
        "recall": float(recall_score(y_val, y_pred, zero_division=0)),
        "f1": float(f1_score(y_val, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": float(fpr),
        "fp_per_hour_est": float(fpr * windows_per_hour),
    }

def evaluate_from_score(y_val: np.ndarray, scores_val: np.ndarray, y_test: np.ndarray, scores_test: np.ndarray, scores_val_normal: np.ndarray | None = None) -> dict:
  # Missing definitions for choose_threshold_normal_percentile and evaluate_with_threshold.
  # Also, y_true, scores are not defined in choose_threshold_max_f1
  # For now, I will use placeholders to allow the code to run without NameErrors
  # and comment out the lines that use undefined functions.
  escolhas = {'max_f1': choose_threshold_max_f1(y_val, scores_val, 0.5)} # Using a placeholder threshold of 0.5
  # if scores_val_normal is not None:
  #   escolhas['p99_5_normal'] = choose_threshold_normal_percentile(scores_val_normal, 99.5)
  out = {'escolhas_de_thresh': escolhas, 'validacao' : {}, 'teste': {}}
  # for name, escolha in escolhas.items():
  #   thr = escolha['threshold']
  #   out['validacao'][name] = evaluate_with_threshold(y_val, scores_val, thr)
  #   out['teste'][name] = evaluate_with_threshold(y_test, scores_test, thr)
  out['primeiro_thresh'] = 'max_f1'
  return out

def bootstrap_auc_pr_ci(y_true: np.ndarray, scores: np.ndarray, n_boot = 500, seed = SEED) -> dict:
  rng = np.random.default_rng(seed)
  vals = []
  n = len(y_true)
  for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    if len(np.unique(y_true[idx])) < 2:
      continue
    vals.append(average_precision_score(y_true[idx], scores[idx]))
  if not vals:
    return {
        'auc_pr_low': None,
        'auc_pr_high': None,
    }
  return{
      'auc_pr_low': float(np.quantile(vals, 0.025)),
      'auc_pr_high': float(np.quantile(vals, 0.975)),
  }

def save_json(obj: dict, path: Path):
  path.parent.mkdir(parents=True, exist_ok=True)
  with open(path, 'w', encoding='utf-8') as f:
    json.dump(obj, f, indent=2)


In [ ]:
#Features
from __future__ import annotations
import numpy as np
from scipy.signal import welch
from scipy.stats import kurtosis, skew
FEATURE_NAMES = [
    "mean",
    "std",
    "rms",
    "energy",
    "abs_peak",
    "peak_to_peak",
    "kurtosis",
    "skewness",
    "zero_crossing_rate",
    "bandpower_0_3hz",
    "bandpower_3_8hz",
    "bandpower_8_15hz",
    "spectral_entropy",
    "spectral_centroid",
    "spectral_rolloff_85",
]

def extract_window_features(x: np.ndarray) -> np.ndarray:
  x = x.astype(np.float32)
  freqs, psd = welch(x, fs = SR, nperseg=min(256, len(x)))
  total_psd = np.sum(psd) + 1e-12
  p = psd/total_psd
  cdf = np.cumsum(p)
  rolloff_idx = int(np.searchsorted(cdf, 0.85))
  centroid = np.sum(freqs * psd) / total_psd
  signs = np.sign(x)
  zcr = np.mean(signs[1:] != signs[:-1])
  return np.array(
    [
        np.mean(x),
        np.std(x),
        np.sqrt(np.mean(x**2)),
        np.sum(x**2),
        np.max(np.abs(x)),
        np.ptp(x),
        kurtosis(x, fisher=True, bias=False),
        skew(x, bias=False),
        zcr,
        np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)]) / total_psd,
        np.sum(psd[(freqs >= 3.0) & (freqs < 8.0)]) / total_psd,
        np.sum(psd[(freqs >= 8.0) & (freqs <= 15.0)]) / total_psd,
        -np.sum(p * np.log(p + 1e-12)),
        centroid,
        freqs[min(rolloff_idx, len(freqs) - 1)],
    ],
    dtype=np.float32,
)
def extract_features(X: np.ndarray, label="") -> np.ndarray:
    print(f"Extracting features {label}: {X.shape}")
    feats = np.vstack([extract_window_features(x) for x in X])
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)


In [ ]:
import tensorflow as tf

def compile_ae(model, lr = 1e-3):
  model.compile(optimizer = tf.keras.optimizers.Adam(lr), loss= 'mse')
  return model
#arquitetura do dense_ae
def build_dense_ae(input_dim = WINDOW_NPTS, latent_dim = 32, lr= 1e-3):
  inputs = tf.keras.layers.Input(shape = (input_dim,))
  x = tf.keras.layers.Dense(256, activation = 'relu')(inputs)
  x = tf.keras.layers.Dense(128, activation = 'relu')(x)
  encoded = tf.keras.layers.Dense(latent_dim, activation = 'relu', name = 'bottleneck')(x)
  x = tf.keras.layers.Dense(128, activation = 'relu')(encoded)
  x = tf.keras.layers.Dense(256, activation = 'relu')(x)
  outputs = tf.keras.layers.Dense(input_dim, activation = 'linear')(x)
  return compile_ae(tf.keras.models.Model(inputs, outputs), lr)

#Arquitetura cnn
def build_cnn1d_ae(input_length=WINDOW_NPTS, lr=1e-3):
    inputs = tf.keras.layers.Input(shape=(input_length, 1))
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 3, padding="same", activation="relu", name="bottleneck")(x)
    x = tf.keras.layers.UpSampling1D(2)(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.UpSampling1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x)
    outputs = tf.keras.layers.Conv1D(1, 7, padding="same", activation="linear")(x)
    return compile_ae(tf.keras.Model(inputs, outputs, name="cnn1d_ae"), lr)

def build_tiny_cnn_ae(input_length=WINDOW_NPTS, lr=1e-3):
    inputs = tf.keras.layers.Input(shape=(input_length, 1))
    x = tf.keras.layers.Conv1D(16, 7, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(16, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(8, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(4, 3, padding="same", activation="relu")(x)
    encoded = tf.keras.layers.MaxPooling1D(2, name="bottleneck")(x)
    x = tf.keras.layers.UpSampling1D(2)(encoded)
    x = tf.keras.layers.Conv1D(4, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.UpSampling1D(2)(x)
    x = tf.keras.layers.Conv1D(8, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.UpSampling1D(2)(x)
    x = tf.keras.layers.Conv1D(16, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.UpSampling1D(2)(x)
    outputs = tf.keras.layers.Conv1D(1, 7, padding="same", activation="linear")(x)
    return compile_ae(tf.keras.Model(inputs, outputs, name="tiny_cnn_ae"), lr)


def build_tiny_cnn_classifier(input_length=WINDOW_NPTS, lr=1e-3):
    inputs = tf.keras.layers.Input(shape=(input_length, 1))
    x = tf.keras.layers.Conv1D(16, 9, padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Conv1D(32, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(16, activation="relu")(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(inputs, outputs, name="tiny_cnn_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="auc_pr"),
            tf.keras.metrics.AUC(curve="ROC", name="auc_roc"),
        ],
    )
    return model

def build_tiny_tcn_classifier(input_length=WINDOW_NPTS, lr=1e-3):
    inputs = tf.keras.layers.Input(shape=(input_length, 1))
    x = tf.keras.layers.Conv1D(16, 7, padding="same", activation="relu")(inputs)
    for dilation in (1, 2, 4, 8):
        residual = x
        x = tf.keras.layers.Conv1D(
            16, 5, padding="same", dilation_rate=dilation, activation="relu"
        )(x)
        x = tf.keras.layers.Conv1D(16, 1, padding="same")(x)
        x = tf.keras.layers.Add()([x, residual])
        x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(inputs, outputs, name="tiny_tcn_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="auc_pr"),
            tf.keras.metrics.AUC(curve="ROC", name="auc_roc"),
        ],
    )
    return model

def build_gru_ae(timesteps=50, features=16, units=64, lr=1e-3):
    inputs = tf.keras.layers.Input(shape=(timesteps, features))
    encoded = tf.keras.layers.GRU(units, return_sequences=False, name="bottleneck")(inputs)
    x = tf.keras.layers.RepeatVector(timesteps)(encoded)
    x = tf.keras.layers.GRU(units, return_sequences=True)(x)
    outputs = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(features))(x)
    return compile_ae(tf.keras.Model(inputs, outputs, name="gru_ae"), lr)


def build_lstm_ae(timesteps=50, features=16, lr=1e-3):
    inputs = tf.keras.layers.Input(shape=(timesteps, features))
    x = tf.keras.layers.LSTM(128, return_sequences=True)(inputs)
    encoded = tf.keras.layers.LSTM(64, return_sequences=False, name="bottleneck")(x)
    x = tf.keras.layers.RepeatVector(timesteps)(encoded)
    x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
    x = tf.keras.layers.LSTM(128, return_sequences=True)(x)
    outputs = tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(features))(x)
    return compile_ae(tf.keras.Model(inputs, outputs, name="lstm_ae"), lr)

## Treinamento ensemble

In [ ]:
from __future__ import annotations

import argparse
import time

import joblib
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Re-running after fix to ensure ValueError is resolved
def build_model(kind: str):
    if kind == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )
    elif kind == "extra_trees":
        clf = ExtraTreesClassifier(
            n_estimators=700,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown model: {kind}")

    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


def main():
    # Simulate command-line arguments for Colab environment
    # Original: parser.add_argument("--model", choices=["random_forest", "extra_trees"], required=True)
    # args = parser.parse_args()
    class Args:
        def __init__(self):
            self.model = "extra_trees" # Setting a default model for execution
    args = Args()

    out_dir = model_dir(args.model)
    ds = load_dataset(DATASET_V4)
    print("Dataset:", DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train_f = extract_features(ds["X_train"], "train")
    X_val_f = extract_features(ds["X_val"], "val")
    X_test_f = extract_features(ds["X_test"], "test")

    model = build_model(args.model)
    t0 = time.time()
    model.fit(X_train_f, ds["y_train"])
    train_seconds = time.time() - t0

    scores_val = model.predict_proba(X_val_f)[:, 1]
    scores_test = model.predict_proba(X_test_f)[:, 1]
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])

    eval_out = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)
    clf = model.named_steps["clf"]
    importances = sorted(
        zip(FEATURE_NAMES, clf.feature_importances_),
        key=lambda item: item[1],
        reverse=True,
    )

    model_path = out_dir / f"{args.model}.joblib"
    joblib.dump(model, model_path)

    result = {
        "model": args.model,
        "family": "features_ml",
        "dataset": str(DATASET_V4),
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "features": FEATURE_NAMES,
        "feature_importances": [
            {"feature": name, "importance": float(value)}
            for name, value in importances
        ],
        "metrics": eval_out,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", eval_out.get("teste", {}).get("max_f1", "N/A (placeholder evaluation)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


Dataset: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}
Extracting features train: (120429, 800)
Extracting features val: (25802, 800)
Extracting features test: (25829, 800)
TEST max_f1: N/A (placeholder evaluation)
Saved: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/extra_trees/result.json


##STA

In [ ]:
from __future__ import annotations

import argparse
import csv
import json
import os
import time
from pathlib import Path
from typing import Any

import numpy as np

try:
    from sklearn.metrics import (
        average_precision_score,
        precision_recall_curve,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
    )
except ImportError as exc:
    raise ImportError(
        "Este script precisa do scikit-learn. No Colab, rode: !pip install scikit-learn"
    ) from exc


# =============================================================================
# CONFIGURAÇÃO PARA GOOGLE COLAB
# =============================================================================

# Monta o Google Drive automaticamente quando estiver no Colab.
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except Exception:
    pass

SEED = 42
SR = 40.0
WINDOW_NPTS = 800
WINDOW_SECONDS = 20.0
STEP_SECONDS = 10.0

# Caminho base no Google Drive.
# Altere aqui se sua pasta tiver outro nome.
BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

# Pasta dos datasets processados.
DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()

# Pasta onde os resultados serão salvos.
RESULTS_DIR = Path(
    os.environ.get(
        "TCC_RESULTS_DIR",
        str(BASE_DIR / "results" / "experimentos_v4"),
    )
).expanduser()

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "dataset_v4_split_evento.npz"


# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def save_json(obj: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    """Calcula média móvel simples mantendo o tamanho do sinal."""
    window = max(int(window), 1)

    if len(x) == 0:
        return x

    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(x, kernel, mode="same")


def compute_scores(
    X: np.ndarray,
    sta_seconds: float,
    lta_seconds: float,
    use_abs: bool,
    name: str = "split",
) -> np.ndarray:
    """
    Calcula o score STA/LTA de cada janela.

    Score usado:
        maior razão STA/LTA dentro da janela.

    Se use_abs=True, usa |x| como função característica.
    Se use_abs=False, usa x², que representa energia do sinal.
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 3:
        X = X.reshape(X.shape[0], -1)

    sta_n = max(int(round(sta_seconds * SR)), 1)
    lta_n = max(int(round(lta_seconds * SR)), sta_n + 1)

    print(
        f"Calculando scores {name}: X={X.shape} | "
        f"STA={sta_seconds}s ({sta_n} amostras) | "
        f"LTA={lta_seconds}s ({lta_n} amostras) | use_abs={use_abs}"
    )

    scores = np.zeros(len(X), dtype=np.float32)
    eps = 1e-8

    for i, x in enumerate(X):
        x = np.asarray(x, dtype=np.float32)

        if use_abs:
            characteristic = np.abs(x)
        else:
            characteristic = x ** 2

        sta = moving_average(characteristic, sta_n)
        lta = moving_average(characteristic, lta_n)
        ratio = sta / (lta + eps)

        scores[i] = float(np.max(ratio))

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


def safe_auc_roc(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Evita erro quando só existe uma classe no split."""
    if len(np.unique(y_true)) < 2:
        return 0.0
    return float(roc_auc_score(y_true, scores))


def summarize_at_threshold(
    y_true: np.ndarray,
    scores: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    y_pred = (scores >= threshold).astype(np.int8)

    return {
        "threshold": float(threshold),
        "auc_pr": float(average_precision_score(y_true, scores)),
        "auc_roc": safe_auc_roc(y_true, scores),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


def find_best_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    best_idx = int(np.nanargmax(f1))

    if len(thresholds) == 0:
        threshold = float(np.max(scores) + 1.0)
    else:
        threshold_idx = min(best_idx, len(thresholds) - 1)
        threshold = float(thresholds[threshold_idx])

    return summarize_at_threshold(y_true, scores, threshold)


def evaluate_from_scores(
    y_val: np.ndarray,
    scores_val: np.ndarray,
    y_test: np.ndarray,
    scores_test: np.ndarray,
) -> dict[str, dict[str, dict[str, float]]]:
    """
    Escolhe o melhor threshold na validação e aplica o mesmo threshold no teste.
    """
    y_val = np.asarray(y_val, dtype=np.int8)
    y_test = np.asarray(y_test, dtype=np.int8)
    scores_val = np.asarray(scores_val, dtype=np.float32)
    scores_test = np.asarray(scores_test, dtype=np.float32)

    val_best = find_best_threshold(y_val, scores_val)
    threshold = val_best["threshold"]
    test_same_threshold = summarize_at_threshold(y_test, scores_test, threshold)

    # Também calcula o melhor F1 possível no teste apenas como diagnóstico.
    # Para comparação justa, use principalmente test["max_f1_val_threshold"].
    test_oracle = find_best_threshold(y_test, scores_test)

    return {
        "validation": {
            "max_f1": val_best,
        },
        "test": {
            "max_f1": test_same_threshold,
            "oracle_max_f1": test_oracle,
        },
    }


def run_single_config(
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    sta_seconds: float,
    lta_seconds: float,
    use_abs: bool,
) -> dict[str, Any]:
    scores_val = compute_scores(
        X_val,
        sta_seconds=sta_seconds,
        lta_seconds=lta_seconds,
        use_abs=use_abs,
        name="val",
    )
    scores_test = compute_scores(
        X_test,
        sta_seconds=sta_seconds,
        lta_seconds=lta_seconds,
        use_abs=use_abs,
        name="test",
    )
    metrics = evaluate_from_scores(y_val, scores_val, y_test, scores_test)

    return {
        "sta_seconds": sta_seconds,
        "lta_seconds": lta_seconds,
        "use_abs": use_abs,
        "scores_val": scores_val,
        "scores_test": scores_test,
        "metrics": metrics,
    }


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", default=str(DATASET_PATH))
    parser.add_argument("--out-dir", default=str(RESULTS_DIR / "stalta_v4"))
    parser.add_argument("--sta", type=float, default=1.0)
    parser.add_argument("--lta", type=float, default=10.0)
    parser.add_argument("--use-abs", action="store_true")
    parser.add_argument(
        "--grid-search",
        action="store_true",
        help="Testa varias combinacoes de STA/LTA na validacao e salva a melhor.",
    )

    # parse_known_args evita erro no Colab/Jupyter por argumentos internos do notebook.
    args, unknown = parser.parse_known_args()
    if unknown:
        print("Argumentos ignorados pelo Colab/Jupyter:", unknown)

    dataset_path = Path(args.dataset).expanduser()
    out_dir = Path(args.out_dir).expanduser()
    out_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("BASELINE CLASSICO STA/LTA")
    print("=" * 80)
    print("DATA_DIR:", DATA_DIR)
    print("RESULTS_DIR:", RESULTS_DIR)
    print("Dataset:", dataset_path)
    print("Saida:", out_dir)

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em:\n{dataset_path}\n\n"
            "Verifique se o arquivo dataset_v4_split_evento.npz está em:\n"
            f"{DATA_DIR}\n\n"
            "Estrutura esperada no Drive:\n"
            "Meu Drive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz"
        )

    data = np.load(dataset_path)

    required_keys = ["X_val", "y_val", "X_test", "y_test"]
    missing = [key for key in required_keys if key not in data.files]
    if missing:
        raise KeyError(
            f"O dataset não possui as chaves esperadas: {missing}\n"
            f"Chaves encontradas: {data.files}"
        )

    X_val = data["X_val"].astype(np.float32)
    y_val = data["y_val"].astype(np.int8)
    X_test = data["X_test"].astype(np.float32)
    y_test = data["y_test"].astype(np.int8)

    print()
    print("Distribuicao:")
    for name, y in [("val", y_val), ("test", y_test)]:
        n_pos = int((y == 1).sum())
        n_total = len(y)
        baseline = n_pos / n_total if n_total else 0.0
        print(
            f"{name:5s}: total={n_total} "
            f"evento={n_pos} "
            f"baseline_auc_pr={baseline:.4f}"
        )

    t0 = time.time()

    if args.grid_search:
        sta_grid = [0.25, 0.5, 1.0, 2.0]
        lta_grid = [4.0, 6.0, 8.0, 10.0, 15.0]
        use_abs_grid = [False, True]

        candidates = []
        best = None

        for sta_seconds in sta_grid:
            for lta_seconds in lta_grid:
                if lta_seconds <= sta_seconds:
                    continue

                for use_abs in use_abs_grid:
                    candidate = run_single_config(
                        X_val,
                        y_val,
                        X_test,
                        y_test,
                        sta_seconds=sta_seconds,
                        lta_seconds=lta_seconds,
                        use_abs=use_abs,
                    )
                    val_metric = candidate["metrics"]["validation"]["max_f1"]
                    row = {
                        "sta_seconds": sta_seconds,
                        "lta_seconds": lta_seconds,
                        "use_abs": use_abs,
                        "val_auc_pr": val_metric["auc_pr"],
                        "val_auc_roc": val_metric["auc_roc"],
                        "val_f1": val_metric["f1"],
                        "val_precision": val_metric["precision"],
                        "val_recall": val_metric["recall"],
                        "threshold": val_metric["threshold"],
                    }
                    candidates.append(row)

                    if best is None:
                        best = candidate
                    else:
                        best_val = best["metrics"]["validation"]["max_f1"]["auc_pr"]
                        if val_metric["auc_pr"] > best_val:
                            best = candidate

        if best is None:
            raise RuntimeError("Nenhuma configuracao valida de STA/LTA encontrada.")

        grid_path = out_dir / "grid_search_stalta.csv"
        with open(grid_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(candidates[0]))
            writer.writeheader()
            writer.writerows(candidates)

        print("Grid search salvo em:", grid_path)
        final = best
    else:
        final = run_single_config(
            X_val,
            y_val,
            X_test,
            y_test,
            sta_seconds=args.sta,
            lta_seconds=args.lta,
            use_abs=args.use_abs,
        )

    elapsed = time.time() - t0

    np.save(out_dir / "scores_val.npy", final["scores_val"])
    np.save(out_dir / "scores_test.npy", final["scores_test"])
    np.save(out_dir / "y_val.npy", y_val)
    np.save(out_dir / "y_test.npy", y_test)

    result = {
        "model": "stalta",
        "family": "classic_signal_processing",
        "dataset": str(dataset_path),
        "score_definition": "max_STA_LTA_ratio_per_window",
        "sample_rate": SR,
        "window_npts": WINDOW_NPTS,
        "window_seconds": WINDOW_SECONDS,
        "step_seconds": STEP_SECONDS,
        "sta_seconds": final["sta_seconds"],
        "lta_seconds": final["lta_seconds"],
        "use_abs": final["use_abs"],
        "train_seconds": 0.0,
        "eval_seconds": elapsed,
        "params": 0,
        "metrics": final["metrics"],
    }
    save_json(result, out_dir / "result.json")

    print()
    print("=" * 80)
    print("RESULTADO STA/LTA")
    print("=" * 80)
    print(json.dumps(result["metrics"]["test"]["max_f1"], indent=2, ensure_ascii=False))
    print()
    print("Configuracao final:")
    print(f"STA: {final['sta_seconds']} s")
    print(f"LTA: {final['lta_seconds']} s")
    print(f"use_abs: {final['use_abs']}")
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Argumentos ignorados pelo Colab/Jupyter: ['-f', '/root/.local/share/jupyter/runtime/kernel-f63d6164-a6ba-47b3-9259-85b4b9547e8d.json']
BASELINE CLASSICO STA/LTA
DATA_DIR: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed
RESULTS_DIR: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/results/experimentos_v4
Dataset: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
Saida: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/results/experimentos_v4/stalta_v4

Distribuicao:
val  : total=25802 evento=3302 baseline_auc_pr=0.1280
test : total=25829 evento=3329 baseline_auc_pr=0.1289
Calculando scores val: X=(25802, 800) | STA=1.0s (40 amostras) | LTA=10.0s (400 amostras) | use_abs=False
Calculando scores test: X=(25829, 800) | STA=1.0s (40 amostras) | LTA=10.0s (400 amostras) | use_abs=False

RESULTADO STA/LTA
{
  "thres

In [ ]:
##Grid Search

from __future__ import annotations

import argparse
import csv
import json
import os
import time
from pathlib import Path
from typing import Any

import numpy as np

try:
    from sklearn.metrics import (
        average_precision_score,
        precision_recall_curve,
        precision_score,
        recall_score,
        f1_score,
        roc_auc_score,
    )
except ImportError as exc:
    raise ImportError(
        "Este script precisa do scikit-learn. No Colab, rode: !pip install scikit-learn"
    ) from exc


# =============================================================================
# CONFIGURAÇÃO PARA GOOGLE COLAB
# =============================================================================

# Monta o Google Drive automaticamente quando estiver no Colab.
try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
except Exception:
    pass

SEED = 42
SR = 40.0
WINDOW_NPTS = 800
WINDOW_SECONDS = 20.0
STEP_SECONDS = 10.0

# Caminho base no Google Drive.
# Altere aqui se sua pasta tiver outro nome.
BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

# Pasta dos datasets processados.
DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()

# Pasta onde os resultados serão salvos.
RESULTS_DIR = Path(
    os.environ.get(
        "TCC_RESULTS_DIR",
        str(BASE_DIR / "results" / "experimentos_v4"),
    )
).expanduser()

DATA_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "dataset_v4_split_evento.npz"


# =============================================================================
# FUNÇÕES AUXILIARES
# =============================================================================

def save_json(obj: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def parse_float_grid(value: str) -> list[float]:
    """
    Converte uma string do tipo "0.25,0.5,1,2" em lista de floats.
    Isso permite alterar o grid direto no terminal/Colab sem mexer no código.
    """
    values = []
    for item in value.split(","):
        item = item.strip()
        if item:
            values.append(float(item))
    if not values:
        raise ValueError(f"Grid vazio ou inválido: {value}")
    return values


def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    """Calcula média móvel simples mantendo o tamanho do sinal."""
    window = max(int(window), 1)

    if len(x) == 0:
        return x

    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(x, kernel, mode="same")


def compute_scores(
    X: np.ndarray,
    sta_seconds: float,
    lta_seconds: float,
    use_abs: bool,
    name: str = "split",
) -> np.ndarray:
    """
    Calcula o score STA/LTA de cada janela.

    Score usado:
        maior razão STA/LTA dentro da janela.

    Se use_abs=True, usa |x| como função característica.
    Se use_abs=False, usa x², que representa energia do sinal.
    """
    X = np.asarray(X, dtype=np.float32)

    if X.ndim == 3:
        X = X.reshape(X.shape[0], -1)

    sta_n = max(int(round(sta_seconds * SR)), 1)
    lta_n = max(int(round(lta_seconds * SR)), sta_n + 1)

    print(
        f"Calculando scores {name}: X={X.shape} | "
        f"STA={sta_seconds}s ({sta_n} amostras) | "
        f"LTA={lta_seconds}s ({lta_n} amostras) | use_abs={use_abs}"
    )

    scores = np.zeros(len(X), dtype=np.float32)
    eps = 1e-8

    for i, x in enumerate(X):
        x = np.asarray(x, dtype=np.float32)

        if use_abs:
            characteristic = np.abs(x)
        else:
            characteristic = x ** 2

        sta = moving_average(characteristic, sta_n)
        lta = moving_average(characteristic, lta_n)
        ratio = sta / (lta + eps)

        scores[i] = float(np.max(ratio))

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


def safe_auc_roc(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Evita erro quando só existe uma classe no split."""
    if len(np.unique(y_true)) < 2:
        return 0.0
    return float(roc_auc_score(y_true, scores))


def safe_auc_pr(y_true: np.ndarray, scores: np.ndarray) -> float:
    """Calcula AUC-PR com proteção para casos degenerados."""
    if len(y_true) == 0:
        return 0.0
    return float(average_precision_score(y_true, scores))


def summarize_at_threshold(
    y_true: np.ndarray,
    scores: np.ndarray,
    threshold: float,
) -> dict[str, float]:
    y_pred = (scores >= threshold).astype(np.int8)

    return {
        "threshold": float(threshold),
        "auc_pr": safe_auc_pr(y_true, scores),
        "auc_roc": safe_auc_roc(y_true, scores),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }


def find_best_threshold(y_true: np.ndarray, scores: np.ndarray) -> dict[str, float]:
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    best_idx = int(np.nanargmax(f1))

    if len(thresholds) == 0:
        threshold = float(np.max(scores) + 1.0)
    else:
        threshold_idx = min(best_idx, len(thresholds) - 1)
        threshold = float(thresholds[threshold_idx])

    return summarize_at_threshold(y_true, scores, threshold)


def evaluate_from_scores(
    y_val: np.ndarray,
    scores_val: np.ndarray,
    y_test: np.ndarray,
    scores_test: np.ndarray,
) -> dict[str, dict[str, dict[str, float]]]:
    """
    Escolhe o melhor threshold na validação e aplica o mesmo threshold no teste.
    """
    y_val = np.asarray(y_val, dtype=np.int8)
    y_test = np.asarray(y_test, dtype=np.int8)
    scores_val = np.asarray(scores_val, dtype=np.float32)
    scores_test = np.asarray(scores_test, dtype=np.float32)

    val_best = find_best_threshold(y_val, scores_val)
    threshold = val_best["threshold"]
    test_same_threshold = summarize_at_threshold(y_test, scores_test, threshold)

    # Também calcula o melhor F1 possível no teste apenas como diagnóstico.
    # Para comparação justa, use principalmente test["max_f1"], que usa threshold da validação.
    test_oracle = find_best_threshold(y_test, scores_test)

    return {
        "validation": {
            "max_f1": val_best,
        },
        "test": {
            "max_f1": test_same_threshold,
            "oracle_max_f1": test_oracle,
        },
    }


def run_validation_config(
    X_val: np.ndarray,
    y_val: np.ndarray,
    sta_seconds: float,
    lta_seconds: float,
    use_abs: bool,
) -> dict[str, Any]:
    """
    Roda somente validação.
    Isso deixa o grid search mais correto e mais rápido, porque o teste só é usado no final.
    """
    scores_val = compute_scores(
        X_val,
        sta_seconds=sta_seconds,
        lta_seconds=lta_seconds,
        use_abs=use_abs,
        name="val",
    )
    val_metrics = find_best_threshold(y_val, scores_val)

    return {
        "sta_seconds": sta_seconds,
        "lta_seconds": lta_seconds,
        "use_abs": use_abs,
        "scores_val": scores_val,
        "validation_metrics": val_metrics,
    }


def run_single_config(
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    sta_seconds: float,
    lta_seconds: float,
    use_abs: bool,
) -> dict[str, Any]:
    scores_val = compute_scores(
        X_val,
        sta_seconds=sta_seconds,
        lta_seconds=lta_seconds,
        use_abs=use_abs,
        name="val",
    )
    scores_test = compute_scores(
        X_test,
        sta_seconds=sta_seconds,
        lta_seconds=lta_seconds,
        use_abs=use_abs,
        name="test",
    )
    metrics = evaluate_from_scores(y_val, scores_val, y_test, scores_test)

    return {
        "sta_seconds": sta_seconds,
        "lta_seconds": lta_seconds,
        "use_abs": use_abs,
        "scores_val": scores_val,
        "scores_test": scores_test,
        "metrics": metrics,
    }


def select_best_candidate(candidates: list[dict[str, Any]], metric: str) -> dict[str, Any]:
    """
    Seleciona a melhor configuração usando apenas a validação.

    metric pode ser:
        - f1
        - auc_pr
        - auc_roc
        - precision
        - recall
    """
    if not candidates:
        raise RuntimeError("Nenhuma configuração válida de STA/LTA encontrada.")

    valid_metrics = {"f1", "auc_pr", "auc_roc", "precision", "recall"}
    if metric not in valid_metrics:
        raise ValueError(f"Métrica inválida: {metric}. Use uma destas: {sorted(valid_metrics)}")

    return max(candidates, key=lambda item: item["validation_metrics"][metric])


def save_grid_csv(candidates: list[dict[str, Any]], path: Path, sort_metric: str) -> None:
    """Salva o resultado do grid search em CSV, ordenado pela melhor métrica de validação."""
    rows = []

    sorted_candidates = sorted(
        candidates,
        key=lambda item: item["validation_metrics"][sort_metric],
        reverse=True,
    )

    for candidate in sorted_candidates:
        val_metric = candidate["validation_metrics"]
        rows.append(
            {
                "rank": len(rows) + 1,
                "sta_seconds": candidate["sta_seconds"],
                "lta_seconds": candidate["lta_seconds"],
                "use_abs": candidate["use_abs"],
                "selection_metric": sort_metric,
                "val_auc_pr": val_metric["auc_pr"],
                "val_auc_roc": val_metric["auc_roc"],
                "val_f1": val_metric["f1"],
                "val_precision": val_metric["precision"],
                "val_recall": val_metric["recall"],
                "threshold": val_metric["threshold"],
            }
        )

    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--dataset", default=str(DATASET_PATH))
    parser.add_argument("--out-dir", default=str(RESULTS_DIR / "stalta_v4"))

    # Configuração manual, usada quando --grid-search não for passado.
    parser.add_argument("--sta", type=float, default=1.0)
    parser.add_argument("--lta", type=float, default=10.0)
    parser.add_argument("--use-abs", action="store_true")

    # Grid search.
    parser.add_argument(
        "--grid-search",
        action="store_true",
        help="Testa várias combinações de STA/LTA na validação e salva a melhor.",
    )
    parser.add_argument(
        "--sta-grid",
        type=str,
        default="0.25,0.5,1.0,1.5,2.0,3.0",
        help="Lista de valores STA em segundos. Exemplo: 0.25,0.5,1,2",
    )
    parser.add_argument(
        "--lta-grid",
        type=str,
        default="4.0,6.0,8.0,10.0,12.0,15.0,20.0",
        help="Lista de valores LTA em segundos. Exemplo: 4,6,8,10,15",
    )
    parser.add_argument(
        "--select-metric",
        type=str,
        default="f1",
        choices=["f1", "auc_pr", "auc_roc", "precision", "recall"],
        help="Métrica de validação usada para escolher a melhor configuração.",
    )

    # parse_known_args evita erro no Colab/Jupyter por argumentos internos do notebook.
    args, unknown = parser.parse_known_args()
    if unknown:
        print("Argumentos ignorados pelo Colab/Jupyter:", unknown)

    dataset_path = Path(args.dataset).expanduser()
    out_dir = Path(args.out_dir).expanduser()
    out_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("BASELINE CLÁSSICO STA/LTA")
    print("=" * 80)
    print("DATA_DIR:", DATA_DIR)
    print("RESULTS_DIR:", RESULTS_DIR)
    print("Dataset:", dataset_path)
    print("Saída:", out_dir)

    if not dataset_path.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em:\n{dataset_path}\n\n"
            "Verifique se o arquivo dataset_v4_split_evento.npz está em:\n"
            f"{DATA_DIR}\n\n"
            "Estrutura esperada no Drive:\n"
            "Meu Drive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz"
        )

    data = np.load(dataset_path)

    required_keys = ["X_val", "y_val", "X_test", "y_test"]
    missing = [key for key in required_keys if key not in data.files]
    if missing:
        raise KeyError(
            f"O dataset não possui as chaves esperadas: {missing}\n"
            f"Chaves encontradas: {data.files}"
        )

    X_val = data["X_val"].astype(np.float32)
    y_val = data["y_val"].astype(np.int8)
    X_test = data["X_test"].astype(np.float32)
    y_test = data["y_test"].astype(np.int8)

    print()
    print("Distribuição:")
    for name, y in [("val", y_val), ("test", y_test)]:
        n_pos = int((y == 1).sum())
        n_total = len(y)
        baseline = n_pos / n_total if n_total else 0.0
        print(
            f"{name:5s}: total={n_total} "
            f"evento={n_pos} "
            f"baseline_auc_pr={baseline:.4f}"
        )

    t0 = time.time()

    if args.grid_search:
        sta_grid = parse_float_grid(args.sta_grid)
        lta_grid = parse_float_grid(args.lta_grid)
        use_abs_grid = [False, True]

        print()
        print("=" * 80)
        print("GRID SEARCH STA/LTA")
        print("=" * 80)
        print("STA grid:", sta_grid)
        print("LTA grid:", lta_grid)
        print("use_abs grid:", use_abs_grid)
        print("Métrica de seleção:", args.select_metric)
        print()

        candidates = []
        total_configs = sum(
            1
            for sta_seconds in sta_grid
            for lta_seconds in lta_grid
            if lta_seconds > sta_seconds
            for _ in use_abs_grid
        )
        config_idx = 0

        for sta_seconds in sta_grid:
            for lta_seconds in lta_grid:
                if lta_seconds <= sta_seconds:
                    continue

                for use_abs in use_abs_grid:
                    config_idx += 1
                    print(
                        f"\nConfiguração {config_idx}/{total_configs}: "
                        f"STA={sta_seconds}s | LTA={lta_seconds}s | use_abs={use_abs}"
                    )

                    candidate = run_validation_config(
                        X_val=X_val,
                        y_val=y_val,
                        sta_seconds=sta_seconds,
                        lta_seconds=lta_seconds,
                        use_abs=use_abs,
                    )
                    candidates.append(candidate)

                    val_metric = candidate["validation_metrics"]
                    print(
                        "Validação | "
                        f"AUC-PR={val_metric['auc_pr']:.4f} | "
                        f"AUC-ROC={val_metric['auc_roc']:.4f} | "
                        f"F1={val_metric['f1']:.4f} | "
                        f"Precision={val_metric['precision']:.4f} | "
                        f"Recall={val_metric['recall']:.4f} | "
                        f"Threshold={val_metric['threshold']:.6f}"
                    )

        best_candidate = select_best_candidate(candidates, args.select_metric)

        grid_path = out_dir / "grid_search_stalta.csv"
        save_grid_csv(candidates, grid_path, args.select_metric)

        best_config = {
            "selection_metric": args.select_metric,
            "sta_seconds": best_candidate["sta_seconds"],
            "lta_seconds": best_candidate["lta_seconds"],
            "use_abs": best_candidate["use_abs"],
            "validation_metrics": best_candidate["validation_metrics"],
        }
        save_json(best_config, out_dir / "best_config.json")

        print()
        print("=" * 80)
        print("MELHOR CONFIGURAÇÃO ENCONTRADA NO GRID SEARCH")
        print("=" * 80)
        print(json.dumps(best_config, indent=2, ensure_ascii=False))
        print("Grid search salvo em:", grid_path)
        print("Melhor configuração salva em:", out_dir / "best_config.json")

        # Agora sim avalia o teste usando apenas a melhor configuração escolhida na validação.
        final = run_single_config(
            X_val=X_val,
            y_val=y_val,
            X_test=X_test,
            y_test=y_test,
            sta_seconds=best_candidate["sta_seconds"],
            lta_seconds=best_candidate["lta_seconds"],
            use_abs=best_candidate["use_abs"],
        )
    else:
        final = run_single_config(
            X_val=X_val,
            y_val=y_val,
            X_test=X_test,
            y_test=y_test,
            sta_seconds=args.sta,
            lta_seconds=args.lta,
            use_abs=args.use_abs,
        )

    elapsed = time.time() - t0

    np.save(out_dir / "scores_val.npy", final["scores_val"])
    np.save(out_dir / "scores_test.npy", final["scores_test"])
    np.save(out_dir / "y_val.npy", y_val)
    np.save(out_dir / "y_test.npy", y_test)

    result = {
        "model": "stalta",
        "family": "classic_signal_processing",
        "dataset": str(dataset_path),
        "score_definition": "max_STA_LTA_ratio_per_window",
        "sample_rate": SR,
        "window_npts": WINDOW_NPTS,
        "window_seconds": WINDOW_SECONDS,
        "step_seconds": STEP_SECONDS,
        "sta_seconds": final["sta_seconds"],
        "lta_seconds": final["lta_seconds"],
        "use_abs": final["use_abs"],
        "train_seconds": 0.0,
        "eval_seconds": elapsed,
        "params": 0,
        "grid_search_enabled": bool(args.grid_search),
        "selection_metric": args.select_metric if args.grid_search else None,
        "metrics": final["metrics"],
    }
    save_json(result, out_dir / "result.json")

    print()
    print("=" * 80)
    print("RESULTADO FINAL STA/LTA NO TESTE")
    print("=" * 80)
    print("Métricas usando threshold escolhido na validação:")
    print(json.dumps(result["metrics"]["test"]["max_f1"], indent=2, ensure_ascii=False))
    print()
    print("Métricas oracle no teste apenas para diagnóstico:")
    print(json.dumps(result["metrics"]["test"]["oracle_max_f1"], indent=2, ensure_ascii=False))
    print()
    print("Configuração final:")
    print(f"STA: {final['sta_seconds']} s")
    print(f"LTA: {final['lta_seconds']} s")
    print(f"use_abs: {final['use_abs']}")
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Argumentos ignorados pelo Colab/Jupyter: ['-f', '/root/.local/share/jupyter/runtime/kernel-4be064cf-c1f2-4875-b386-2555cc6891af.json']
BASELINE CLÁSSICO STA/LTA
DATA_DIR: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed
RESULTS_DIR: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/results/experimentos_v4
Dataset: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
Saída: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/results/experimentos_v4/stalta_v4

Distribuição:
val  : total=25802 evento=3302 baseline_auc_pr=0.1280
test : total=25829 evento=3329 baseline_auc_pr=0.1289
Calculando scores val: X=(25802, 800) | STA=1.0s (40 amostras) | LTA=10.0s (400 amostras) | use_abs=False
Calculando scores test: X=(25829, 800) | STA=1.0s (40 amostras) | LTA=10.0s (400 amostras) | use_abs=False

RESULTADO FINAL STA/LTA NO T

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path

# Re-define BASE_DIR and RESULT_DIR as they were globally defined in other cells
BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

# Define paths for different result directories based on where models saved their outputs
RESULTS_DIR_OLD = BASE_DIR / "results" # Used by RF, ET, STA/LTA initial runs
RESULTS_DIR_NEW = BASE_DIR / "resultados" # Used by Keras and Optuna models (via model_dir)

def load_result_json(path: Path):
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        # print(f"Warning: Result file not found at {path}")
        return None
    except json.JSONDecodeError:
        # print(f"Warning: Error decoding JSON from {path}")
        return None

# List of models and their result file paths
model_configs = [
    {
        "Modelo": "STA/LTA",
        "Tipo": "clássico",
        "path": RESULTS_DIR_OLD / "experimentos_v4" / "stalta_v4" / "result.json",
        "auc_pr_key": ["metrics", "test", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "test", "max_f1", "f1"],
    },
    {
        "Modelo": "Random Forest",
        "Tipo": "ML features",
        "path": RESULTS_DIR_OLD / "rf" / "resultado_rf.json",
        "auc_pr_key": ["teste", "average_precision"],
        "f1_key": ["teste", "f1"],
    },
    {
        "Modelo": "ExtraTrees",
        "Tipo": "ML features",
        "path": RESULTS_DIR_OLD / "et" / "resultado_et.json",
        "auc_pr_key": ["teste", "average_precision"],
        "f1_key": ["teste", "f1"],
    },
    {
        "Modelo": "Dense AE",
        "Tipo": "DL baseline",
        "path": None, # No explicit run for Dense AE in provided cells
        "auc_pr_key": None,
        "f1_key": None,
    },
    {
        "Modelo": "CNN 1D AE antiga",
        "Tipo": "DL AE",
        "path": RESULTS_DIR_NEW / "cnn1d_ae" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
    {
        "Modelo": "Tiny CNN AE",
        "Tipo": "TinyML AE",
        "path": None, # No explicit run for Tiny CNN AE in provided cells
        "auc_pr_key": None,
        "f1_key": None,
    },
    {
        "Modelo": "Tiny CNN classifier",
        "Tipo": "TinyML sup.",
        "path": RESULTS_DIR_NEW / "tiny_cnn_classifier" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
    {
        "Modelo": "Tiny CNN classifier (Optuna)",
        "Tipo": "TinyML sup.",
        "path": RESULTS_DIR_NEW / "optuna_tiny_cnn_classifier" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
    {
        "Modelo": "Tiny TCN classifier",
        "Tipo": "TinyML sup.",
        "path": RESULTS_DIR_NEW / "tiny_tcn_classifier" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
    {
        "Modelo": "Tiny TCN classifier (Optuna)",
        "Tipo": "TinyML sup.",
        "path": RESULTS_DIR_NEW / "optuna_tiny_tcn_classifier" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
    {
        "Modelo": "GRU AE",
        "Tipo": "DL seq. leve",
        "path": None,
        "auc_pr_key": None,
        "f1_key": None,
    },
    {
        "Modelo": "LSTM AE",
        "Tipo": "DL seq. pesado",
        "path": None,
        "auc_pr_key": None,
        "f1_key": None,
    },
    {
        "Modelo": "Random Forest (Optuna)",
        "Tipo": "ML features",
        "path": RESULTS_DIR_NEW / "optuna_random_forest" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
    {
        "Modelo": "ExtraTrees (Optuna)",
        "Tipo": "ML features",
        "path": RESULTS_DIR_NEW / "optuna_extra_trees" / "result.json",
        "auc_pr_key": ["metrics", "escolhas_de_thresh", "max_f1", "auc_pr"],
        "f1_key": ["metrics", "escolhas_de_thresh", "max_f1", "f1"],
    },
]

results_data = []

for config in model_configs:
    model_name = config["Modelo"]
    model_type = config["Tipo"]
    auc_pr = "N/A"
    f1 = "N/A"

    if config["path"] and config["path"].exists():
        result = load_result_json(config["path"])
        if result:
            if config["auc_pr_key"]:
                current_val = result
                for key in config["auc_pr_key"]:
                    if isinstance(current_val, dict) and key in current_val:
                        current_val = current_val[key]
                    else:
                        current_val = "N/A"
                        break
                auc_pr = round(current_val, 4) if isinstance(current_val, (int, float)) else current_val

            if config["f1_key"]:
                current_val = result
                for key in config["f1_key"]:
                    if isinstance(current_val, dict) and key in current_val:
                        current_val = current_val[key]
                    else:
                        current_val = "N/A"
                        break
                f1 = round(current_val, 4) if isinstance(current_val, (int, float)) else current_val

    results_data.append({
        "Modelo": model_name,
        "Tipo": model_type,
        "AUC-PR": auc_pr,
        "F1": f1,
        "KB": "N/A",
        "ms": "N/A",
        "Int8": "N/A",
        "OTA": "N/A",
    })

results_df = pd.DataFrame(results_data)

display(results_df)

,Modelo,Tipo,AUC-PR,F1,KB,ms,Int8,OTA
0,STA/LTA,clássico,0.1642,0.2721,N/A,N/A,N/A,N/A
1,Random Forest,ML features,0.7473,0.6749,N/A,N/A,N/A,N/A
2,ExtraTrees,ML features,0.7374,0.6645,N/A,N/A,N/A,N/A
3,Dense AE,DL baseline,N/A,N/A,N/A,N/A,N/A,N/A
4,CNN 1D AE antiga,DL AE,0.1404,0.0,N/A,N/A,N/A,N/A
5,Tiny CNN AE,TinyML AE,N/A,N/A,N/A,N/A,N/A,N/A
6,Tiny CNN classifier,TinyML sup.,0.8897,0.7959,N/A,N/A,N/A,N/A
7,Tiny CNN classifier (Optuna),TinyML sup.,0.902,0.8189,N/A,N/A,N/A,N/A
8,Tiny TCN classifier,TinyML sup.,0.8688,0.7791,N/A,N/A,N/A,N/A
9,Tiny TCN classifier (Optuna),TinyML sup.,0.9186,0.8247,N/A,N/A,N/A,N/A


traino autoencoder

In [ ]:
import  argparse
import time
import numpy as np
import tensorflow as tf
def set_seed():
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


def score_reconstruction(model, X_in: np.ndarray, X_flat: np.ndarray, batch_size=2048):
    rec = model.predict(X_in, batch_size=batch_size, verbose=0)
    return np.mean((X_flat - rec.reshape(-1, WINDOW_NPTS)) ** 2, axis=1)


def prepare_inputs(kind: str, X: np.ndarray):
    if kind == "dense_ae":
        return X
    if kind in {"cnn1d_ae", "tiny_cnn_ae"}:
        return to_cnn(X)
    if kind in {"gru_ae", "lstm_ae"}:
        return to_seq(X, 50, 16)
    raise ValueError(kind)


def build(kind: str, lr: float):
    if kind == "dense_ae":
        return build_dense_ae(lr=lr)
    if kind == "cnn1d_ae":
        return build_cnn1d_ae(lr=lr)
    if kind == "tiny_cnn_ae":
        return build_tiny_cnn_ae(lr=lr)
    if kind == "gru_ae":
        return build_gru_ae(lr=lr)
    if kind == "lstm_ae":
        return build_lstm_ae(lr=lr)
    raise ValueError(kind)


def main():
    # Simulate command-line arguments for Colab environment
    class Args:
        def __init__(self):
            self.model = "cnn1d_ae" # Setting a default model for execution
            self.epochs = 120
            self.batch_size = 512
            self.lr = 1e-3
    args = Args()

    set_seed()
    out_dir = model_dir(args.model)
    ds = load_dataset(DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train_n, _ = split_normal_event(ds["X_train"], ds["y_train"])
    X_val_n, X_val_a = split_normal_event(ds["X_val"], ds["y_val"])
    X_test_n, X_test_a = split_normal_event(ds["X_test"], ds["y_test"])

    model = build(args.model, args.lr)
    train_in = prepare_inputs(args.model, X_train_n)
    val_in = prepare_inputs(args.model, X_val_n)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=6, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log.csv")),
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "checkpoint.keras"), monitor="val_loss", save_best_only=True
        ),
    ]

    t0 = time.time()
    history = model.fit(
        train_in,
        train_in,
        validation_data=(val_in, val_in),
        epochs=args.epochs,
        batch_size=args.batch_size,
        callbacks=callbacks,
        verbose=1,
    )
    train_seconds = time.time() - t0

    s_val_n = score_reconstruction(model, prepare_inputs(args.model, X_val_n), X_val_n)
    s_val_a = score_reconstruction(model, prepare_inputs(args.model, X_val_a), X_val_a)
    s_test_n = score_reconstruction(model, prepare_inputs(args.model, X_test_n), X_test_n)
    s_test_a = score_reconstruction(model, prepare_inputs(args.model, X_test_a), X_test_a)

    y_val = np.concatenate([np.zeros(len(s_val_n)), np.ones(len(s_val_a))]).astype(np.int8)
    scores_val = np.concatenate([s_val_n, s_val_a])
    y_test = np.concatenate([np.zeros(len(s_test_n)), np.ones(len(s_test_a))]).astype(np.int8)
    scores_test = np.concatenate([s_test_n, s_test_a])
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", y_val)
    np.save(out_dir / "y_test.npy", y_test)
    eval_out = evaluate_from_score(y_val, scores_val, y_test, scores_test, s_val_n)

    model_path = out_dir / f"{args.model}.keras"
    model.save(model_path)
    result = {
        "model": args.model,
        "family": "autoencoder",
        "dataset": str(DATASET_V4),
        "params": model.count_params(),
        "epochs_ran": len(history.history["loss"]),
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "metrics": eval_out,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", eval_out.get("test", {}).get("max_f1", "N/A (placeholder)"))
    print("TEST p99_5_normal:", eval_out.get("test", {}).get("p99_5_normal", "N/A (placeholder)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}
Epoch 1/120
206/206 ━━━━━━━━━━━━━━━━━━━━ 34s 128ms/step - loss: 0.1866 - val_loss: 0.0296 - learning_rate: 0.0010
Epoch 2/120
206/206 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - loss: 0.0201 - val_loss: 0.0138 - learning_rate: 0.0010
Epoch 3/120
206/206 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - loss: 0.0120 - val_loss: 0.0104 - learning_rate: 0.0010
Epoch 4/120
206/206 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - loss: 0.0090 - val_loss: 0.0081 - learning_rate: 0.0010
Epoch 5/120
206/206 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - loss: 0.0074 - val_loss: 0.0062 - learning_rate: 0.0010
Epoch 6/120
206/206 ━━━━━━━━━━━━━━━━━━━━ 22s 107ms/step - loss: 0.0062 - val_loss: 0.0053 - learning_rate: 0.0010
Epoch 7/120
206/206 ━━━━━━━━━━━━

traino classificadores

In [ ]:
from __future__ import annotations

import argparse
import time

import numpy as np
import tensorflow as tf

BUILDERS = {
    "tiny_cnn_classifier": build_tiny_cnn_classifier,
    "tiny_tcn_classifier": build_tiny_tcn_classifier,
}


def set_seed():
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


def main():
    # Simulate command-line arguments for Colab environment
    class Args:
        def __init__(self):
            self.model = "tiny_tcn_classifier" # Setting a default model for execution
            self.epochs = 120
            self.batch_size = 512
            self.lr = 1e-3
    args = Args()

    set_seed()
    out_dir = model_dir(args.model)
    ds = load_dataset(DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train = to_cnn(ds["X_train"])
    X_val = to_cnn(ds["X_val"])
    X_test = to_cnn(ds["X_test"])

    model = BUILDERS[args.model](WINDOW_NPTS, lr=args.lr)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc_pr", mode="max", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc_pr", mode="max", factor=0.5, patience=6, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log.csv")),
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "checkpoint.keras"),
            monitor="val_auc_pr",
            mode="max",
            save_best_only=True,
        ),
    ]

    t0 = time.time()
    history = model.fit(
        X_train,
        ds["y_train"],
        validation_data=(X_val, ds["y_val"]),
        epochs=args.epochs,
        batch_size=args.batch_size,
        class_weight=make_class_weights(ds["y_train"]),
        callbacks=callbacks,
        verbose=1,
    )
    train_seconds = time.time() - t0

    scores_val = model.predict(X_val, batch_size=2048, verbose=0).ravel()
    scores_test = model.predict(X_test, batch_size=2048, verbose=0).ravel()
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])
    eval_out = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)

    model_path = out_dir / f"{args.model}.keras"
    model.save(model_path)
    result = {
        "model": args.model,
        "family": "deep_classifier",
        "dataset": str(DATASET_V4),
        "params": model.count_params(),
        "epochs_ran": len(history.history["loss"]),
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "metrics": eval_out,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", eval_out.get("teste", {}).get("max_f1", "N/A (placeholder evaluation)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}
Epoch 1/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 42s 121ms/step - auc_pr: 0.5144 - auc_roc: 0.8099 - loss: 0.5290 - val_auc_pr: 0.6447 - val_auc_roc: 0.8553 - val_loss: 0.3573 - learning_rate: 0.0010
Epoch 2/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 19s 79ms/step - auc_pr: 0.6723 - auc_roc: 0.8671 - loss: 0.4330 - val_auc_pr: 0.6782 - val_auc_roc: 0.8764 - val_loss: 0.4345 - learning_rate: 0.0010
Epoch 3/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 19s 79ms/step - auc_pr: 0.7039 - auc_roc: 0.8840 - loss: 0.4061 - val_auc_pr: 0.7144 - val_auc_roc: 0.8860 - val_loss: 0.4367 - learning_rate: 0.0010
Epoch 4/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 19s 79ms/step - auc_pr: 0.7270 - auc_roc: 0.8935 - loss: 0.3914 - val_auc_pr: 0.7323 - val_auc_roc: 0

In [ ]:
from __future__ import annotations

import argparse
import time

import numpy as np
import tensorflow as tf

BUILDERS = {
    "tiny_cnn_classifier": build_tiny_cnn_classifier,
    "tiny_tcn_classifier": build_tiny_tcn_classifier,
}


def set_seed():
    np.random.seed(SEED)
    tf.random.set_seed(SEED)


def main():
    # Simulate command-line arguments for Colab environment
    class Args:
        def __init__(self):
            self.model = "tiny_cnn_classifier" # Setting a default model for execution
            self.epochs = 120
            self.batch_size = 512
            self.lr = 1e-3
    args = Args()

    set_seed()
    out_dir = model_dir(args.model)
    ds = load_dataset(DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train = to_cnn(ds["X_train"])
    X_val = to_cnn(ds["X_val"])
    X_test = to_cnn(ds["X_test"])

    model = BUILDERS[args.model](WINDOW_NPTS, lr=args.lr)
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc_pr", mode="max", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc_pr", mode="max", factor=0.5, patience=6, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log.csv")),
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "checkpoint.keras"),
            monitor="val_auc_pr",
            mode="max",
            save_best_only=True,
        ),
    ]

    t0 = time.time()
    history = model.fit(
        X_train,
        ds["y_train"],
        validation_data=(X_val, ds["y_val"]),
        epochs=args.epochs,
        batch_size=args.batch_size,
        class_weight=make_class_weights(ds["y_train"]),
        callbacks=callbacks,
        verbose=1,
    )
    train_seconds = time.time() - t0

    scores_val = model.predict(X_val, batch_size=2048, verbose=0).ravel()
    scores_test = model.predict(X_test, batch_size=2048, verbose=0).ravel()
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])
    eval_out = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)

    model_path = out_dir / f"{args.model}.keras"
    model.save(model_path)
    result = {
        "model": args.model,
        "family": "deep_classifier",
        "dataset": str(DATASET_V4),
        "params": model.count_params(),
        "epochs_ran": len(history.history["loss"]),
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "metrics": eval_out,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", eval_out.get("test", {}).get("max_f1", "N/A (placeholder)"))
    print("TEST p99_5_normal:", eval_out.get("test", {}).get("p99_5_normal", "N/A (placeholder)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}
Epoch 1/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 19s 67ms/step - auc_pr: 0.5868 - auc_roc: 0.8253 - loss: 0.5000 - val_auc_pr: 0.6515 - val_auc_roc: 0.8507 - val_loss: 0.4210 - learning_rate: 0.0010
Epoch 2/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 51ms/step - auc_pr: 0.6759 - auc_roc: 0.8652 - loss: 0.4344 - val_auc_pr: 0.6766 - val_auc_roc: 0.8712 - val_loss: 0.3947 - learning_rate: 0.0010
Epoch 3/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 51ms/step - auc_pr: 0.7047 - auc_roc: 0.8816 - loss: 0.4128 - val_auc_pr: 0.7121 - val_auc_roc: 0.8854 - val_loss: 0.3226 - learning_rate: 0.0010
Epoch 4/120
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 51ms/step - auc_pr: 0.7356 - auc_roc: 0.8924 - loss: 0.3930 - val_auc_pr: 0.7340 - val_auc_roc: 0.

optuna


In [ ]:
import argparse
import time
import joblib
import numpy as np
import optuna
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import os
import json
from scipy.signal import welch
from scipy.stats import kurtosis, skew

# --- Definitions from Lo7L2yi9xMsb ---
SEED = 42
SR = 40.0
WINDOW_NPTS = 800
WINDOW_SECONDS = 20.0
STEP_SECONDS = 10.0

BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()
RESULTS_DIR = BASE_DIR / "resultados"
DATA_DIR.mkdir(parents = True, exist_ok = True)
DATASET_V4 = DATA_DIR/'dataset_v4_split_evento.npz'

def model_dir(name: str) -> Path:
  path = RESULTS_DIR / name
  path.mkdir(parents=True, exist_ok=True)
  return path

# --- Definitions from hKZhHEjCyGLX ---
def load_dataset(path=DATASET_V4) -> dict[str, np.ndarray]:
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em:\n{path}\n\n"
            "Verifique se o arquivo dataset_v4_split_evento.npz está na pasta processed do Google Drive."
        )
    data = np.load(path)
    required_keys = [
        "X_train", "y_train",
        "X_val", "y_val",
        "X_test", "y_test",
    ]
    missing = [key for key in required_keys if key not in data.files]
    if missing:
        raise KeyError(
            f"O arquivo .npz não possui as chaves esperadas: {missing}\n"
            f"Chaves encontradas: {data.files}"
        )
    return {
        "X_train": data["X_train"].astype(np.float32),
        "y_train": data["y_train"].astype(np.int8),
        "X_val": data["X_val"].astype(np.float32),
        "y_val": data["y_val"].astype(np.int8),
        "X_test": data["X_test"].astype(np.float32),
        "y_test": data["y_test"].astype(np.int8),
    }

def class_summary(y: np.ndarray) -> dict:
    y = np.asarray(y)
    total = int(len(y))
    normal = int((y == 0).sum())
    event = int((y == 1).sum())
    return {
        "total": total,
        "normal": normal,
        "event": event,
        "baseline_auc_pr": event / total if total else 0.0,
    }

# --- Definitions from P7p5pwvBkJFb ---
FEATURE_NAMES = [
    "mean",
    "std",
    "rms",
    "energy",
    "abs_peak",
    "peak_to_peak",
    "kurtosis",
    "skewness",
    "zero_crossing_rate",
    "bandpower_0_3hz",
    "bandpower_3_8hz",
    "bandpower_8_15hz",
    "spectral_entropy",
    "spectral_centroid",
    "spectral_rolloff_85",
]

def extract_window_features(x: np.ndarray) -> np.ndarray:
  x = x.astype(np.float32)
  # SR comes from Lo7L2yi9xMsb
  freqs, psd = welch(x, fs = SR, nperseg=min(256, len(x)))
  total_psd = np.sum(psd) + 1e-12
  p = psd/total_psd
  cdf = np.cumsum(p)
  rolloff_idx = int(np.searchsorted(cdf, 0.85))
  centroid = np.sum(freqs * psd) / total_psd
  signs = np.sign(x)
  zcr = np.mean(signs[1:] != signs[:-1])
  return np.array(
    [
        np.mean(x),
        np.std(x),
        np.sqrt(np.mean(x**2)),
        np.sum(x**2),
        np.max(np.abs(x)),
        np.ptp(x),
        kurtosis(x, fisher=True, bias=False),
        skew(x, bias=False),
        zcr,
        np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)]) / total_psd,
        np.sum(psd[(freqs >= 3.0) & (freqs < 8.0)]) / total_psd,
        np.sum(psd[(freqs >= 8.0) & (freqs <= 15.0)]) / total_psd,
        -np.sum(p * np.log(p + 1e-12)),
        centroid,
        freqs[min(rolloff_idx, len(freqs) - 1)],
    ],
    dtype=np.float32,
)
def extract_features(X: np.ndarray, label="") -> np.ndarray:
    print(f"Extracting features {label}: {X.shape}")
    feats = np.vstack([extract_window_features(x) for x in X])
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

# --- Definitions from _2_NKSI0W52Yl ---
def choose_threshold_max_f1(y_val: np.ndarray, scores_val: np.ndarray, threshold: float) ->dict :
  y_pred = (scores_val >= threshold).astype(np.int8)
  tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels = [0,1]).ravel()
  fpr = fp/max(tn+fp,1) # fpr = false positive rate
  # STEP_SECONDS comes from Lo7L2yi9xMsb
  windows_per_hour = 3600 / STEP_SECONDS
  return {
        "auc_pr": float(average_precision_score(y_val, scores_val)),
        "auc_roc": float(roc_auc_score(y_val, scores_val)),
        "threshold": float(threshold),
        "precision": float(precision_score(y_val, y_pred, zero_division=0)),
        "recall": float(recall_score(y_val, y_pred, zero_division=0)),
        "f1": float(f1_score(y_val, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": float(fpr),
        "fp_per_hour_est": float(fpr * windows_per_hour),
    }

def evaluate_from_score(y_val: np.ndarray, scores_val: np.ndarray, y_test: np.ndarray, scores_test: np.ndarray, scores_val_normal: np.ndarray | None = None) -> dict:
  escolhas = {'max_f1': choose_threshold_max_f1(y_val, scores_val, 0.5)} # Using a placeholder threshold of 0.5
  out = {'escolhas_de_thresh': escolhas, 'validacao' : {}, 'teste': {}}
  out['primeiro_thresh'] = 'max_f1'
  return out

def bootstrap_auc_pr_ci(y_true: np.ndarray, scores: np.ndarray, n_boot = 500, seed = SEED) -> dict:
  rng = np.random.default_rng(seed)
  vals = []
  n = len(y_true)
  for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    if len(np.unique(y_true[idx])) < 2:
      continue
    vals.append(average_precision_score(y_true[idx], scores[idx]))
  if not vals:
    return {
        'auc_pr_low': None,
        'auc_pr_high': None,
    }
  return{
      'auc_pr_low': float(np.quantile(vals, 0.025)),
      'auc_pr_high': float(np.quantile(vals, 0.975)),
  }

def save_json(obj: dict, path: Path):
  path.parent.mkdir(parents=True, exist_ok=True)
  with open(path, 'w', encoding='utf-8') as f:
    json.dump(obj, f, indent=2)


def build_model(trial: optuna.Trial, kind: str):
    n_estimators = trial.suggest_int("n_estimators", 300, 1200, step=100)
    max_depth = trial.suggest_categorical("max_depth", [None, 8, 12, 16, 24, 32])
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 8)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 16)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    if kind == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )
    elif kind == "extra_trees":
        clf = ExtraTreesClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown model: {kind}")

    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


def main():
    # Simulate command-line arguments for Colab environment
    class Args:
        def __init__(self):
            self.model = "random_forest"  # Set a default model
            self.trials = 40
            self.study_name = None
    args = Args()

    study_name = args.study_name or f"optuna_{args.model}"
    out_dir = model_dir(study_name)

    ds = load_dataset(DATASET_V4)
    print("Dataset:", DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train_f = extract_features(ds["X_train"], "train")
    X_val_f = extract_features(ds["X_val"], "val")
    X_test_f = extract_features(ds["X_test"], "test")

    def objective(trial: optuna.Trial):
        model = build_model(trial, args.model)
        model.fit(X_train_f, ds["y_train"])
        scores_val = model.predict_proba(X_val_f)[:, 1]
        return float(average_precision_score(ds["y_val"], scores_val))

    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=args.trials)

    study.trials_dataframe().to_csv(out_dir / "trials.csv", index=False)
    save_json(
        {
            "study_name": study_name,
            "model": args.model,
            "best_value": float(study.best_value),
            "best_params": study.best_params,
        },
        out_dir / "best_trial.json",
    )

    print("Best params:", study.best_params)
    print("Best value:", study.best_value)

    best_trial = optuna.trial.FixedTrial(study.best_params)
    best_model = build_model(best_trial, args.model)
    t0 = time.time()
    best_model.fit(X_train_f, ds["y_train"])
    train_seconds = time.time() - t0

    scores_val = best_model.predict_proba(X_val_f)[:, 1]
    scores_test = best_model.predict_proba(X_test_f)[:, 1]
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])

    metrics = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)
    clf = best_model.named_steps["clf"]
    importances = sorted(
        zip(FEATURE_NAMES, clf.feature_importances_),
        key=lambda item: item[1],
        reverse=True,
    )

    model_path = out_dir / f"{study_name}.joblib"
    joblib.dump(best_model, model_path)

    result = {
        "model": study_name,
        "base_model": args.model,
        "family": "features_ml_optuna",
        "dataset": str(DATASET_V4),
        "best_params": study.best_params,
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "features": FEATURE_NAMES,
        "feature_importances": [
            {"feature": name, "importance": float(value)}
            for name, value in importances
        ],
        "metrics": metrics,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("max_f1", {}).get("f1", "N/A (placeholder evaluation)"))
    print("TEST p99_5_normal:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("p99_5_normal", {}).get("f1", "N/A (placeholder evaluation)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()

Dataset: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}
Extracting features train: (120429, 800)
Extracting features val: (25802, 800)
Extracting features test: (25829, 800)


[I 2026-05-19 14:30:28,419] A new study created in memory with name: optuna_random_forest
[I 2026-05-19 14:34:36,618] Trial 0 finished with value: 0.7627280496581513 and parameters: {'n_estimators': 600, 'max_depth': None, 'min_samples_leaf': 7, 'min_samples_split': 11, 'max_features': None}. Best is trial 0 with value: 0.7627280496581513.
[I 2026-05-19 14:42:20,255] Trial 1 finished with value: 0.7621509683834685 and parameters: {'n_estimators': 1100, 'max_depth': 24, 'min_samples_leaf': 3, 'min_samples_split': 11, 'max_features': None}. Best is trial 0 with value: 0.7627280496581513.
[I 2026-05-19 14:43:31,146] Trial 2 finished with value: 0.7616662038447202 and parameters: {'n_estimators': 700, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'max_features': 'log2'}. Best is trial 0 with value: 0.7627280496581513.
[I 2026-05-19 14:44:07,655] Trial 3 finished with value: 0.6901676413832105 and parameters: {'n_estimators': 600, 'max_depth': 8, 'min_samples_leaf': 8, '

Best params: {'n_estimators': 900, 'max_depth': 24, 'min_samples_leaf': 5, 'min_samples_split': 10, 'max_features': None}
Best value: 0.7641164193892281
TEST max_f1: N/A (placeholder evaluation)
TEST p99_5_normal: N/A (placeholder evaluation)
Saved: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/optuna_random_forest/result.json


In [ ]:
import argparse
import time
import joblib
import numpy as np
import optuna
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, precision_score, recall_score,
    f1_score, confusion_matrix, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import os
import json
from scipy.signal import welch
from scipy.stats import kurtosis, skew

# --- Definitions from Lo7L2yi9xMsb ---
SEED = 42
SR = 40.0
WINDOW_NPTS = 800
WINDOW_SECONDS = 20.0
STEP_SECONDS = 10.0

BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()

DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()
RESULTS_DIR = BASE_DIR / "resultados"
DATA_DIR.mkdir(parents = True, exist_ok = True)
DATASET_V4 = DATA_DIR/'dataset_v4_split_evento.npz'

def model_dir(name: str) -> Path:
  path = RESULTS_DIR / name
  path.mkdir(parents=True, exist_ok=True)
  return path

# --- Definitions from hKZhHEjCyGLX ---
def load_dataset(path=DATASET_V4) -> dict[str, np.ndarray]:
    path = Path(path).expanduser()
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em:\n{path}\n\n"
            "Verifique se o arquivo dataset_v4_split_evento.npz está na pasta processed do Google Drive."
        )
    data = np.load(path)
    required_keys = [
        "X_train", "y_train",
        "X_val", "y_val",
        "X_test", "y_test",
    ]
    missing = [key for key in required_keys if key not in data.files]
    if missing:
        raise KeyError(
            f"O arquivo .npz não possui as chaves esperadas: {missing}\n"
            f"Chaves encontradas: {data.files}"
        )
    return {
        "X_train": data["X_train"].astype(np.float32),
        "y_train": data["y_train"].astype(np.int8),
        "X_val": data["X_val"].astype(np.float32),
        "y_val": data["y_val"].astype(np.int8),
        "X_test": data["X_test"].astype(np.float32),
        "y_test": data["y_test"].astype(np.int8),
    }

def class_summary(y: np.ndarray) -> dict:
    y = np.asarray(y)
    total = int(len(y))
    normal = int((y == 0).sum())
    event = int((y == 1).sum())
    return {
        "total": total,
        "normal": normal,
        "event": event,
        "baseline_auc_pr": event / total if total else 0.0,
    }

# --- Definitions from P7p5pwvBkJFb ---
FEATURE_NAMES = [
    "mean",
    "std",
    "rms",
    "energy",
    "abs_peak",
    "peak_to_peak",
    "kurtosis",
    "skewness",
    "zero_crossing_rate",
    "bandpower_0_3hz",
    "bandpower_3_8hz",
    "bandpower_8_15hz",
    "spectral_entropy",
    "spectral_centroid",
    "spectral_rolloff_85",
]

def extract_window_features(x: np.ndarray) -> np.ndarray:
  x = x.astype(np.float32)
  # SR comes from Lo7L2yi9xMsb
  freqs, psd = welch(x, fs = SR, nperseg=min(256, len(x)))
  total_psd = np.sum(psd) + 1e-12
  p = psd/total_psd
  cdf = np.cumsum(p)
  rolloff_idx = int(np.searchsorted(cdf, 0.85))
  centroid = np.sum(freqs * psd) / total_psd
  signs = np.sign(x)
  zcr = np.mean(signs[1:] != signs[:-1])
  return np.array(
    [
        np.mean(x),
        np.std(x),
        np.sqrt(np.mean(x**2)),
        np.sum(x**2),
        np.max(np.abs(x)),
        np.ptp(x),
        kurtosis(x, fisher=True, bias=False),
        skew(x, bias=False),
        zcr,
        np.sum(psd[(freqs >= 0.5) & (freqs < 3.0)]) / total_psd,
        np.sum(psd[(freqs >= 3.0) & (freqs < 8.0)]) / total_psd,
        np.sum(psd[(freqs >= 8.0) & (freqs <= 15.0)]) / total_psd,
        -np.sum(p * np.log(p + 1e-12)),
        centroid,
        freqs[min(rolloff_idx, len(freqs) - 1)],
    ],
    dtype=np.float32,
)
def extract_features(X: np.ndarray, label="") -> np.ndarray:
    print(f"Extracting features {label}: {X.shape}")
    feats = np.vstack([extract_window_features(x) for x in X])
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

# --- Definitions from _2_NKSI0W52Yl ---
def choose_threshold_max_f1(y_val: np.ndarray, scores_val: np.ndarray, threshold: float) ->dict :
  y_pred = (scores_val >= threshold).astype(np.int8)
  tn, fp, fn, tp = confusion_matrix(y_val, y_pred, labels = [0,1]).ravel()
  fpr = fp/max(tn+fp,1) # fpr = false positive rate
  # STEP_SECONDS comes from Lo7L2yi9xMsb
  windows_per_hour = 3600 / STEP_SECONDS
  return {
        "auc_pr": float(average_precision_score(y_val, scores_val)),
        "auc_roc": float(roc_auc_score(y_val, scores_val)),
        "threshold": float(threshold),
        "precision": float(precision_score(y_val, y_pred, zero_division=0)),
        "recall": float(recall_score(y_val, y_pred, zero_division=0)),
        "f1": float(f1_score(y_val, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fpr": float(fpr),
        "fp_per_hour_est": float(fpr * windows_per_hour),
    }

def evaluate_from_score(y_val: np.ndarray, scores_val: np.ndarray, y_test: np.ndarray, scores_test: np.ndarray, scores_val_normal: np.ndarray | None = None) -> dict:
  escolhas = {'max_f1': choose_threshold_max_f1(y_val, scores_val, 0.5)} # Using a placeholder threshold of 0.5
  out = {'escolhas_de_thresh': escolhas, 'validacao' : {}, 'teste': {}}
  out['primeiro_thresh'] = 'max_f1'
  return out

def bootstrap_auc_pr_ci(y_true: np.ndarray, scores: np.ndarray, n_boot = 500, seed = SEED) -> dict:
  rng = np.random.default_rng(seed)
  vals = []
  n = len(y_true)
  for _ in range(n_boot):
    idx = rng.integers(0, n, n)
    if len(np.unique(y_true[idx])) < 2:
      continue
    vals.append(average_precision_score(y_true[idx], scores[idx]))
  if not vals:
    return {
        'auc_pr_low': None,
        'auc_pr_high': None,
    }
  return{
      'auc_pr_low': float(np.quantile(vals, 0.025)),
      'auc_pr_high': float(np.quantile(vals, 0.975)),
  }

def save_json(obj: dict, path: Path):
  path.parent.mkdir(parents=True, exist_ok=True)
  with open(path, 'w', encoding='utf-8') as f:
    json.dump(obj, f, indent=2)


def build_model(trial: optuna.Trial, kind: str):
    n_estimators = trial.suggest_int("n_estimators", 300, 1200, step=100)
    max_depth = trial.suggest_categorical("max_depth", [None, 8, 12, 16, 24, 32])
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 8)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 16)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    if kind == "random_forest":
        clf = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            class_weight="balanced_subsample",
            random_state=SEED,
            n_jobs=-1,
        )
    elif kind == "extra_trees":
        clf = ExtraTreesClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            min_samples_split=min_samples_split,
            max_features=max_features,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1,
        )
    else:
        raise ValueError(f"Unknown model: {kind}")

    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


def main():
    # Simulate command-line arguments for Colab environment
    class Args:
        def __init__(self):
            self.model = "extra_trees"  # Set a default model
            self.trials = 40
            self.study_name = None
    args = Args()

    study_name = args.study_name or f"optuna_{args.model}"
    out_dir = model_dir(study_name)

    ds = load_dataset(DATASET_V4)
    print("Dataset:", DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train_f = extract_features(ds["X_train"], "train")
    X_val_f = extract_features(ds["X_val"], "val")
    X_test_f = extract_features(ds["X_test"], "test")

    def objective(trial: optuna.Trial):
        model = build_model(trial, args.model)
        model.fit(X_train_f, ds["y_train"])
        scores_val = model.predict_proba(X_val_f)[:, 1]
        return float(average_precision_score(ds["y_val"], scores_val))

    study = optuna.create_study(
        direction="maximize",
        study_name=study_name,
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=args.trials)

    study.trials_dataframe().to_csv(out_dir / "trials.csv", index=False)
    save_json(
        {
            "study_name": study_name,
            "model": args.model,
            "best_value": float(study.best_value),
            "best_params": study.best_params,
        },
        out_dir / "best_trial.json",
    )

    print("Best params:", study.best_params)
    print("Best value:", study.best_value)

    best_trial = optuna.trial.FixedTrial(study.best_params)
    best_model = build_model(best_trial, args.model)
    t0 = time.time()
    best_model.fit(X_train_f, ds["y_train"])
    train_seconds = time.time() - t0

    scores_val = best_model.predict_proba(X_val_f)[:, 1]
    scores_test = best_model.predict_proba(X_test_f)[:, 1]
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])

    metrics = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)
    clf = best_model.named_steps["clf"]
    importances = sorted(
        zip(FEATURE_NAMES, clf.feature_importances_),
        key=lambda item: item[1],
        reverse=True,
    )

    model_path = out_dir / f"{study_name}.joblib"
    joblib.dump(best_model, model_path)

    result = {
        "model": study_name,
        "base_model": args.model,
        "family": "features_ml_optuna",
        "dataset": str(DATASET_V4),
        "best_params": study.best_params,
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "features": FEATURE_NAMES,
        "feature_importances": [
            {"feature": name, "importance": float(value)}
            for name, value in importances
        ],
        "metrics": metrics,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("max_f1", {}).get("f1", "N/A (placeholder evaluation)"))
    print("TEST p99_5_normal:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("p99_5_normal", {}).get("f1", "N/A (placeholder evaluation)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()

Dataset: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/processed/dataset_v4_split_evento.npz
train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}
Extracting features train: (120429, 800)
Extracting features val: (25802, 800)
Extracting features test: (25829, 800)


[I 2026-05-20 09:51:39,474] A new study created in memory with name: optuna_extra_trees
[I 2026-05-20 09:52:12,583] Trial 0 finished with value: 0.7701180339278211 and parameters: {'n_estimators': 600, 'max_depth': None, 'min_samples_leaf': 7, 'min_samples_split': 11, 'max_features': None}. Best is trial 0 with value: 0.7701180339278211.
[I 2026-05-20 09:53:15,050] Trial 1 finished with value: 0.7744948946799215 and parameters: {'n_estimators': 1100, 'max_depth': 24, 'min_samples_leaf': 3, 'min_samples_split': 11, 'max_features': None}. Best is trial 1 with value: 0.7744948946799215.
[I 2026-05-20 09:53:31,601] Trial 2 finished with value: 0.7538226503400214 and parameters: {'n_estimators': 700, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'max_features': 'log2'}. Best is trial 1 with value: 0.7744948946799215.
[I 2026-05-20 09:53:37,025] Trial 3 finished with value: 0.5965309085057073 and parameters: {'n_estimators': 600, 'max_depth': 8, 'min_samples_leaf': 8, 'mi

Best params: {'n_estimators': 1000, 'max_depth': 24, 'min_samples_leaf': 2, 'min_samples_split': 4, 'max_features': None}
Best value: 0.7767427947841077
TEST max_f1: N/A (placeholder evaluation)
TEST p99_5_normal: N/A (placeholder evaluation)
Saved: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/optuna_extra_trees/result.json


In [ ]:
from __future__ import annotations

import argparse
import time

import numpy as np
import optuna
import tensorflow as tf

# As variáveis e funcões (DATASET_V4, SEED, WINDOW_NPTS, model_dir,
# class_summary, load_dataset, make_class_weights, to_cnn, evaluate_from_score, save_json)
# já estão no namespace global após a execução das células anteriores.

# Install optuna-integration for tfkeras if not already installed
try:
    import optuna_integration.tfkeras
except ImportError:
    %pip install optuna-integration[tfkeras]

def set_seed(seed: int = SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)


def build_model(trial: optuna.Trial, input_length: int = WINDOW_NPTS):
    n_blocks = trial.suggest_int("n_blocks", 2, 4)
    base_filters = trial.suggest_categorical("base_filters", [8, 12, 16, 24])
    kernel_first = trial.suggest_categorical("kernel_first", [7, 9, 11, 15])
    kernel_other = trial.suggest_categorical("kernel_other", [3, 5, 7, 9])
    dense_units = trial.suggest_categorical("dense_units", [8, 16, 24, 32, 48])
    dropout = trial.suggest_float("dropout", 0.0, 0.45)
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    use_batch_norm = trial.suggest_categorical("use_batch_norm", [True, False])
    pooling = trial.suggest_categorical("pooling", ["max", "average"])

    inputs = tf.keras.layers.Input(shape=(input_length, 1))
    x = inputs

    for i in range(n_blocks):
        filters = min(base_filters * (2**i), 64)
        kernel = kernel_first if i == 0 else kernel_other
        x = tf.keras.layers.Conv1D(
            filters=filters,
            kernel_size=kernel,
            padding="same",
            use_bias=not use_batch_norm,
        )(x)
        if use_batch_norm:
            x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)

        if i < n_blocks - 1:
            if pooling == "max":
                x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
            else:
                x = tf.keras.layers.AveragePooling1D(pool_size=2)(x)

        if dropout > 0:
            x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(dense_units, activation="relu")(x)
    if dropout > 0:
        x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs, name="optuna_tiny_cnn_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="auc_pr"),
            tf.keras.metrics.AUC(curve="ROC", name="auc_roc"),
        ],
    )
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--trials", type=int, default=40)
    parser.add_argument("--epochs", type=int, default=80)
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--max-params", type=int, default=20000)
    parser.add_argument("--study-name", default="optuna_tiny_cnn_classifier")
    args = parser.parse_args(args=[])

    set_seed()
    out_dir = model_dir(args.study_name)
    ds = load_dataset(DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train = to_cnn(ds["X_train"])
    X_val = to_cnn(ds["X_val"])
    X_test = to_cnn(ds["X_test"])
    class_weight = make_class_weights(ds["y_train"])

    def objective(trial: optuna.Trial):
        tf.keras.backend.clear_session()
        set_seed(SEED + trial.number)
        model = build_model(trial)
        params = model.count_params()
        trial.set_user_attr("params", int(params))

        if params > args.max_params:
            raise optuna.TrialPruned(f"params={params} > max_params={args.max_params}")

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_auc_pr",
                mode="max",
                patience=10,
                restore_best_weights=True,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_auc_pr",
                mode="max",
                factor=0.5,
                patience=4,
                min_lr=1e-6,
            ),
            optuna.integration.TFKerasPruningCallback(trial, "val_auc_pr"),
        ]

        history = model.fit(
            X_train,
            ds["y_train"],
            validation_data=(X_val, ds["y_val"]),
            epochs=args.epochs,
            batch_size=args.batch_size,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=0,
        )

        best_auc_pr = float(max(history.history["val_auc_pr"]))
        penalty = 0.002 * (params / args.max_params)
        return best_auc_pr - penalty

    study = optuna.create_study(
        direction="maximize",
        study_name=args.study_name,
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=8),
    )
    study.optimize(objective, n_trials=args.trials)

    trials_df = study.trials_dataframe()
    trials_df.to_csv(out_dir / "trials.csv", index=False)
    save_json(
        {
            "study_name": args.study_name,
            "best_value": float(study.best_value),
            "best_params": study.best_params,
            "best_trial_user_attrs": study.best_trial.user_attrs,
        },
        out_dir / "best_trial.json",
    )

    print("Best params:", study.best_params)
    print("Best value:", study.best_value)

    tf.keras.backend.clear_session()
    set_seed()
    fixed_trial = optuna.trial.FixedTrial(study.best_params)
    best_model = build_model(fixed_trial)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc_pr", mode="max", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc_pr", mode="max", factor=0.5, patience=6, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log_best.csv")),
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "checkpoint_best.keras"),
            monitor="val_auc_pr",
            mode="max",
            save_best_only=True,
        ),
    ]

    t0 = time.time()
    history = best_model.fit(
        X_train,
        ds["y_train"],
        validation_data=(X_val, ds["y_val"]),
        epochs=args.epochs,
        batch_size=args.batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )
    train_seconds = time.time() - t0

    scores_val = best_model.predict(X_val, batch_size=2048, verbose=0).ravel()
    scores_test = best_model.predict(X_test, batch_size=2048, verbose=0).ravel()
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])

    metrics = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)
    model_path = out_dir / "optuna_tiny_cnn_classifier.keras"
    best_model.save(model_path)

    result = {
        "model": "optuna_tiny_cnn_classifier",
        "family": "deep_classifier_optuna",
        "dataset": str(DATASET_V4),
        "best_params": study.best_params,
        "params": int(best_model.count_params()),
        "epochs_ran": len(history.history["loss"]),
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "metrics": metrics,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("max_f1", {}).get("f1", "N/A (placeholder)"))
    print("TEST p99_5_normal:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("p99_5_normal", {}).get("f1", "N/A (placeholder)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


[I 2026-05-19 17:45:11,317] A new study created in memory with name: optuna_tiny_cnn_classifier


train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}


[I 2026-05-19 17:48:04,715] Trial 0 finished with value: 0.819331922454834 and parameters: {'n_blocks': 3, 'base_filters': 8, 'kernel_first': 11, 'kernel_other': 7, 'dense_units': 48, 'dropout': 0.1943752583889521, 'lr': 0.0002692655251486473, 'use_batch_norm': True, 'pooling': 'average'}. Best is trial 0 with value: 0.819331922454834.
[I 2026-05-19 18:02:14,678] Trial 1 finished with value: 0.8770616760398865 and parameters: {'n_blocks': 3, 'base_filters': 8, 'kernel_first': 9, 'kernel_other': 5, 'dense_units': 16, 'dropout': 0.015474834501848278, 'lr': 0.0022038218939289885, 'use_batch_norm': False, 'pooling': 'average'}. Best is trial 1 with value: 0.8770616760398865.
[I 2026-05-19 18:18:06,655] Trial 2 finished with value: 0.896863072228241 and parameters: {'n_blocks': 3, 'base_filters': 12, 'kernel_first': 11, 'kernel_other': 9, 'dense_units': 16, 'dropout': 0.0634159012386432, 'lr': 0.0015308837415731389, 'use_batch_norm': False, 'pooling': 'max'}. Best is trial 2 with value: 0.8

Best params: {'n_blocks': 3, 'base_filters': 16, 'kernel_first': 9, 'kernel_other': 5, 'dense_units': 32, 'dropout': 0.06408689488073668, 'lr': 0.002324567134169603, 'use_batch_norm': False, 'pooling': 'average'}
Best value: 0.8977215076118469
Epoch 1/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 25s 77ms/step - auc_pr: 0.5588 - auc_roc: 0.8350 - loss: 0.4951 - val_auc_pr: 0.6254 - val_auc_roc: 0.8697 - val_loss: 0.3550 - learning_rate: 0.0023
Epoch 2/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - auc_pr: 0.6584 - auc_roc: 0.8753 - loss: 0.4240 - val_auc_pr: 0.6598 - val_auc_roc: 0.8809 - val_loss: 0.3249 - learning_rate: 0.0023
Epoch 3/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - auc_pr: 0.7143 - auc_roc: 0.8911 - loss: 0.3977 - val_auc_pr: 0.7397 - val_auc_roc: 0.9011 - val_loss: 0.4308 - learning_rate: 0.0023
Epoch 4/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 12s 50ms/step - auc_pr: 0.7592 - auc_roc: 0.9047 - loss: 0.3729 - val_auc_pr: 0.7665 - val_auc_roc: 0.9098 - val_loss: 0.4453 - learning_rate: 0.0

In [ ]:
from __future__ import annotations

import argparse
import time

import numpy as np
import optuna
import tensorflow as tf

# As variáveis e funcões necessárias já estão no namespace global.

def set_seed(seed: int = SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)


def residual_tcn_block(x, filters: int, kernel_size: int, dilation: int, dropout: float):
    residual = x
    x = tf.keras.layers.Conv1D(
        filters,
        kernel_size,
        padding="same",
        dilation_rate=dilation,
        use_bias=False,
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    if dropout > 0:
        x = tf.keras.layers.Dropout(dropout)(x)

    x = tf.keras.layers.Conv1D(filters, 1, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    if residual.shape[-1] != filters:
        residual = tf.keras.layers.Conv1D(filters, 1, padding="same")(residual)

    x = tf.keras.layers.Add()([x, residual])
    x = tf.keras.layers.Activation("relu")(x)
    return x


def build_model(trial: optuna.Trial, input_length: int = WINDOW_NPTS):
    filters = trial.suggest_categorical("filters", [8, 12, 16, 24, 32])
    kernel_size = trial.suggest_categorical("kernel_size", [3, 5, 7, 9])
    n_blocks = trial.suggest_int("n_blocks", 3, 5)
    dropout = trial.suggest_float("dropout", 0.0, 0.40)
    dense_units = trial.suggest_categorical("dense_units", [0, 8, 16, 32])
    lr = trial.suggest_float("lr", 1e-4, 3e-3, log=True)

    dilation_choices = {
        3: [1, 2, 4],
        4: [1, 2, 4, 8],
        5: [1, 2, 4, 8, 16],
    }

    inputs = tf.keras.layers.Input(shape=(input_length, 1))
    x = tf.keras.layers.Conv1D(filters, 1, padding="same")(inputs)
    for dilation in dilation_choices[n_blocks]:
        x = residual_tcn_block(x, filters, kernel_size, dilation, dropout)

    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    if dense_units > 0:
        x = tf.keras.layers.Dense(dense_units, activation="relu")(x)
        if dropout > 0:
            x = tf.keras.layers.Dropout(dropout)(x)
    outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

    model = tf.keras.Model(inputs, outputs, name="optuna_tiny_tcn_classifier")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.AUC(curve="PR", name="auc_pr"),
            tf.keras.metrics.AUC(curve="ROC", name="auc_roc"),
        ],
    )
    return model


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--trials", type=int, default=40)
    parser.add_argument("--epochs", type=int, default=80)
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--max-params", type=int, default=20000)
    parser.add_argument("--study-name", default="optuna_tiny_tcn_classifier")
    args = parser.parse_args(args=[]) # Corrigido para evitar erro no notebook

    set_seed()
    out_dir = model_dir(args.study_name)
    ds = load_dataset(DATASET_V4)
    for part in ("train", "val", "test"):
        print(part, class_summary(ds[f"y_{part}"]))

    X_train = to_cnn(ds["X_train"])
    X_val = to_cnn(ds["X_val"])
    X_test = to_cnn(ds["X_test"])
    class_weight = make_class_weights(ds["y_train"])

    def objective(trial: optuna.Trial):
        tf.keras.backend.clear_session()
        set_seed(SEED + trial.number)
        model = build_model(trial)
        params = model.count_params()
        trial.set_user_attr("params", int(params))

        if params > args.max_params:
            raise optuna.TrialPruned(f"params={params} > max_params={args.max_params}")

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_auc_pr",
                mode="max",
                patience=10,
                restore_best_weights=True,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_auc_pr",
                mode="max",
                factor=0.5,
                patience=4,
                min_lr=1e-6,
            ),
            optuna.integration.TFKerasPruningCallback(trial, "val_auc_pr"),
        ]

        history = model.fit(
            X_train,
            ds["y_train"],
            validation_data=(X_val, ds["y_val"]),
            epochs=args.epochs,
            batch_size=args.batch_size,
            class_weight=class_weight,
            callbacks=callbacks,
            verbose=0,
        )
        best_auc_pr = float(max(history.history["val_auc_pr"]))
        penalty = 0.002 * (params / args.max_params)
        return best_auc_pr - penalty

    study = optuna.create_study(
        direction="maximize",
        study_name=args.study_name,
        sampler=optuna.samplers.TPESampler(seed=SEED),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=8),
    )
    study.optimize(objective, n_trials=args.trials)

    study.trials_dataframe().to_csv(out_dir / "trials.csv", index=False)
    save_json(
        {
            "study_name": args.study_name,
            "best_value": float(study.best_value),
            "best_params": study.best_params,
            "best_trial_user_attrs": study.best_trial.user_attrs,
        },
        out_dir / "best_trial.json",
    )

    print("Best params:", study.best_params)
    print("Best value:", study.best_value)

    tf.keras.backend.clear_session()
    set_seed()
    fixed_trial = optuna.trial.FixedTrial(study.best_params)
    best_model = build_model(fixed_trial)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_auc_pr", mode="max", patience=15, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_auc_pr", mode="max", factor=0.5, patience=6, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(out_dir / "training_log_best.csv")),
        tf.keras.callbacks.ModelCheckpoint(
            str(out_dir / "checkpoint_best.keras"),
            monitor="val_auc_pr",
            mode="max",
            save_best_only=True,
        ),
    ]

    t0 = time.time()
    history = best_model.fit(
        X_train,
        ds["y_train"],
        validation_data=(X_val, ds["y_val"]),
        epochs=args.epochs,
        batch_size=args.batch_size,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=1,
    )
    train_seconds = time.time() - t0

    scores_val = best_model.predict(X_val, batch_size=2048, verbose=0).ravel()
    scores_test = best_model.predict(X_test, batch_size=2048, verbose=0).ravel()
    np.save(out_dir / "scores_val.npy", scores_val)
    np.save(out_dir / "scores_test.npy", scores_test)
    np.save(out_dir / "y_val.npy", ds["y_val"])
    np.save(out_dir / "y_test.npy", ds["y_test"])

    metrics = evaluate_from_score(ds["y_val"], scores_val, ds["y_test"], scores_test)
    model_path = out_dir / "optuna_tiny_tcn_classifier.keras"
    best_model.save(model_path)

    result = {
        "model": "optuna_tiny_tcn_classifier",
        "family": "deep_classifier_optuna",
        "dataset": str(DATASET_V4),
        "best_params": study.best_params,
        "params": int(best_model.count_params()),
        "epochs_ran": len(history.history["loss"]),
        "train_seconds": train_seconds,
        "model_path": str(model_path),
        "metrics": metrics,
    }
    save_json(result, out_dir / "result.json")
    print("TEST max_f1:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("max_f1", {}).get("f1", "N/A (placeholder)"))
    print("TEST p99_5_normal:", metrics.get("teste", {}).get("escolhas_de_thresh", {}).get("p99_5_normal", {}).get("f1", "N/A (placeholder)"))
    print("Saved:", out_dir / "result.json")


if __name__ == "__main__":
    main()


train {'total': 120429, 'normal': 105000, 'event': 15429, 'baseline_auc_pr': 0.1281169817901004}
val {'total': 25802, 'normal': 22500, 'event': 3302, 'baseline_auc_pr': 0.12797457561429346}
test {'total': 25829, 'normal': 22500, 'event': 3329, 'baseline_auc_pr': 0.12888613573889815}


[I 2026-05-19 20:41:25,356] A new study created in memory with name: optuna_tiny_tcn_classifier
[I 2026-05-19 21:21:07,666] Trial 0 finished with value: 0.8640907371948242 and parameters: {'filters': 12, 'kernel_size': 7, 'n_blocks': 5, 'dropout': 0.008233797718320978, 'dense_units': 0, 'lr': 0.00018659959624904942}. Best is trial 0 with value: 0.8640907371948242.
[I 2026-05-19 21:21:08,736] Trial 1 pruned. params=53633 > max_params=20000
[I 2026-05-19 21:41:39,262] Trial 2 finished with value: 0.8642149494094848 and parameters: {'filters': 16, 'kernel_size': 5, 'n_blocks': 4, 'dropout': 0.013755408446087358, 'dense_units': 0, 'lr': 0.000586412916969653}. Best is trial 2 with value: 0.8642149494094848.
[I 2026-05-19 21:58:54,546] Trial 3 finished with value: 0.8720072568023681 and parameters: {'filters': 16, 'kernel_size': 7, 'n_blocks': 3, 'dropout': 0.018090915564215226, 'dense_units': 32, 'lr': 0.0003364867144187955}. Best is trial 3 with value: 0.8720072568023681.
[I 2026-05-19 22:

Best params: {'filters': 24, 'kernel_size': 9, 'n_blocks': 3, 'dropout': 0.03750292603024232, 'dense_units': 16, 'lr': 0.0022631059340978083}
Best value: 0.9101020373443603
Epoch 1/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 36s 108ms/step - auc_pr: 0.7545 - auc_roc: 0.9061 - loss: 0.3738 - val_auc_pr: 0.6893 - val_auc_roc: 0.8632 - val_loss: 0.2703 - learning_rate: 0.0023
Epoch 2/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 17s 70ms/step - auc_pr: 0.8201 - auc_roc: 0.9346 - loss: 0.3137 - val_auc_pr: 0.7504 - val_auc_roc: 0.9042 - val_loss: 0.5741 - learning_rate: 0.0023
Epoch 3/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 17s 71ms/step - auc_pr: 0.8367 - auc_roc: 0.9419 - loss: 0.2962 - val_auc_pr: 0.8050 - val_auc_roc: 0.9283 - val_loss: 0.5949 - learning_rate: 0.0023
Epoch 4/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 17s 71ms/step - auc_pr: 0.8483 - auc_roc: 0.9478 - loss: 0.2815 - val_auc_pr: 0.8159 - val_auc_roc: 0.9370 - val_loss: 0.4024 - learning_rate: 0.0023
Epoch 5/80
236/236 ━━━━━━━━━━━━━━━━━━━━ 17s 71ms/step - auc_pr: 0.

## Analise de erros


In [ ]:
import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import os

BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()
RESULTS_DIR = BASE_DIR / "resultados"
WINDOW_NPTS = 800

def main():
    top_k = 10

    # Carrega os dados
    ds = load_dataset()
    X = ds["X_test"]
    y = ds["y_test"]

    # Lista com os modelos rodados no Optuna
    optuna_models = [
        "optuna_random_forest",
        "optuna_extra_trees",
        "optuna_tiny_cnn_classifier",
        "optuna_tiny_tcn_classifier"
    ]

    all_metrics = []

    for model_name in optuna_models:
        print(f"Processando {model_name}...")
        result_json = RESULTS_DIR / model_name / "result.json"
        scores_npy = RESULTS_DIR / model_name / "scores_test.npy"

        if not result_json.exists() or not scores_npy.exists():
            print(f"Arquivos não encontrados para {model_name}. Pulando.\n")
            continue

        scores = np.load(scores_npy)
        with open(result_json, encoding="utf-8") as f:
            result = json.load(f)

        # Ajuste do caminho da métrica conforme o JSON gerado na função evaluate_from_score
        try:
            metrics_node = result.get("metrics", {}).get("escolhas_de_thresh", {}).get("max_f1", {})
            threshold = metrics_node.get("threshold", 0.5)
            auc_pr = metrics_node.get("auc_pr", None)
            f1 = metrics_node.get("f1", None)
        except Exception:
            threshold = 0.5
            auc_pr = None
            f1 = None

        all_metrics.append({
            "Modelo": model_name,
            "Threshold": threshold,
            "AUC-PR": auc_pr,
            "F1": f1
        })

        pred = (scores >= threshold).astype(int)

        out_dir = RESULTS_DIR / "error_analysis" / model_name
        out_dir.mkdir(parents=True, exist_ok=True)

        df = pd.DataFrame({"idx": np.arange(len(y)), "y": y, "score": scores, "pred": pred})
        df["kind"] = np.select(
            [
                (df.y == 0) & (df.pred == 0),
                (df.y == 0) & (df.pred == 1),
                (df.y == 1) & (df.pred == 0),
                (df.y == 1) & (df.pred == 1),
            ],
            ["tn", "fp", "fn", "tp"],
            default="unknown"
        )
        df.to_csv(out_dir / "errors.csv", index=False)

        chosen = pd.concat(
            [
                df[df.kind == "fp"].sort_values("score", ascending=False).head(top_k),
                df[df.kind == "fn"].sort_values("score", ascending=True).head(top_k),
            ]
        )

        for _, row in chosen.iterrows():
            x = X[int(row.idx)]
            plt.figure(figsize=(10, 3))
            plt.plot(np.arange(WINDOW_NPTS) / 40.0, x, linewidth=0.8)
            plt.title(f"{row.kind.upper()} idx={row.idx} score={row.score:.6f}")
            plt.xlabel("Tempo (s)")
            plt.ylabel("Amplitude z-score")
            plt.tight_layout()
            plt.savefig(out_dir / f"{row.kind}_{int(row.idx)}.png", dpi=150)
            plt.close()

        print(f"Análise de erros salva em: {out_dir}\n")

    print("Resumo das Métricas dos Modelos Optuna:")
    display(pd.DataFrame(all_metrics))

if __name__ == "__main__":
    main()


Processando optuna_random_forest...
Análise de erros salva em: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/error_analysis/optuna_random_forest

Processando optuna_extra_trees...
Análise de erros salva em: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/error_analysis/optuna_extra_trees

Processando optuna_tiny_cnn_classifier...
Análise de erros salva em: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/error_analysis/optuna_tiny_cnn_classifier

Processando optuna_tiny_tcn_classifier...
Análise de erros salva em: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/error_analysis/optuna_tiny_tcn_classifier

Resumo das Métricas dos Modelos Optuna:


,Modelo,Threshold,AUC-PR,F1
0,optuna_random_forest,0.5,0.764116,0.689877
1,optuna_extra_trees,0.5,0.776743,0.696426
2,optuna_tiny_cnn_classifier,0.5,0.901969,0.818911
3,optuna_tiny_tcn_classifier,0.5,0.918625,0.824748


## Otimizacao}

In [ ]:
from pathlib import Path
import argparse
import json
import time
import os # Added import for os

import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

# Redefine global paths and constants for self-containment
# These values are taken from previously executed cells like Lo7L2yi9xMsb
BASE_DIR = Path(
    os.environ.get(
        "TCC_BASE_DIR",
        "/content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data",
    )
).expanduser()
DATA_DIR = Path(
    os.environ.get(
        "TCC_PROCESSED_DIR",
        str(BASE_DIR / "processed"),
    )
).expanduser()
RESULTS_DIR = BASE_DIR / "resultados"
DATASET_V4 = DATA_DIR/'dataset_v4_split_evento.npz'
STEP_SECONDS = 10.0

def carregar_dataset(path):
    data = np.load(path)

    return {
        "X_train": data["X_train"].astype(np.float32),
        "y_train": data["y_train"].astype(np.int8),
        "X_val": data["X_val"].astype(np.float32),
        "y_val": data["y_val"].astype(np.int8),
        "X_test": data["X_test"].astype(np.float32),
        "y_test": data["y_test"].astype(np.int8),
    }


def adaptar_entrada(X, input_shape):
    shape_esperado = tuple(dim for dim in input_shape[1:] if dim is not None)

    if X.shape[1:] == shape_esperado:
        return X.astype(np.float32)

    return X.reshape((-1, *shape_esperado)).astype(np.float32)


def representative_dataset(X_train, input_shape, n_amostras):
    X = adaptar_entrada(X_train[:n_amostras], input_shape)

    def gen():
        for i in range(len(X)):
            yield [X[i : i + 1].astype(np.float32)]

    return gen


def escolher_threshold_validacao(y_val, scores_val):
    precision, recall, thresholds = precision_recall_curve(y_val, scores_val)

    f1 = 2 * precision * recall / (precision + recall + 1e-12)

    idx = int(np.argmax(f1[:-1]))

    return {
        "threshold": float(thresholds[idx]),
        "precision_val": float(precision[idx]),
        "recall_val": float(recall[idx]),
        "f1_val": float(f1[idx]),
    }


def avaliar_threshold(y_true, scores, threshold, step_seconds):
    y_pred = (scores >= threshold).astype(np.int8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    fpr = fp / max(fp + tn, 1)
    windows_per_hour = 3600.0 / step_seconds

    return {
        "auc_pr": float(average_precision_score(y_true, scores)),
        "auc_roc": float(roc_auc_score(y_true, scores)),
        "threshold": float(threshold),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "fp_per_hour": float(fpr * windows_per_hour),
    }


def avaliar_com_validacao(y_val, scores_val, y_test, scores_test, step_seconds):
    escolha = escolher_threshold_validacao(y_val, scores_val)

    threshold = escolha["threshold"]

    return {
        "threshold_escolhido_na_validacao": escolha,
        "validacao": avaliar_threshold(
            y_val,
            scores_val,
            threshold,
            step_seconds,
        ),
        "teste": avaliar_threshold(
            y_test,
            scores_test,
            threshold,
            step_seconds,
        ),
    }


def predizer_keras(model, X, input_shape, batch_size):
    X_in = adaptar_entrada(X, input_shape)

    scores = model.predict(
        X_in,
        batch_size=batch_size,
        verbose=0,
    )

    return scores.ravel().astype(np.float32)


def converter_float32(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    return converter.convert()


def converter_dynamic_range(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    return converter.convert()


def converter_float16(model):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    return converter.convert()


def converter_int8(model, X_train, input_shape, n_amostras):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    converter.representative_dataset = representative_dataset(
        X_train,
        input_shape,
        n_amostras,
    )

    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    ]

    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8

    return converter.convert()


def quantizar_entrada(sample, input_detail):
    dtype = input_detail["dtype"]

    if dtype == np.float32:
        return sample.astype(np.float32)

    scale, zero_point = input_detail["quantization"]

    q = np.round(sample / scale + zero_point)

    info = np.iinfo(dtype)

    q = np.clip(q, info.min, info.max)

    return q.astype(dtype)


def dequantizar_saida(output, output_detail):
    dtype = output_detail["dtype"]

    if dtype == np.float32:
        return output.astype(np.float32)

    scale, zero_point = output_detail["quantization"]

    return (output.astype(np.float32) - zero_point) * scale


def predizer_tflite(tflite_path, X, input_shape, limite=None):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))

    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    X_in = adaptar_entrada(X, input_shape)

    if limite is not None:
        X_in = X_in[:limite]

    scores = []
    tempos_ms = []

    for i in range(len(X_in)):
        sample = X_in[i : i + 1]

        sample_quantizado = quantizar_entrada(sample, input_detail)

        interpreter.set_tensor(input_detail["index"], sample_quantizado)

        inicio = time.perf_counter()

        interpreter.invoke()

        tempo_ms = (time.perf_counter() - inicio) * 1000.0

        output = interpreter.get_tensor(output_detail["index"])

        output = dequantizar_saida(output, output_detail)

        scores.append(float(output.ravel()[0]))

        tempos_ms.append(tempo_ms)

    return np.array(scores, dtype=np.float32), {
        "n": int(len(tempos_ms)),
        "mean_ms": float(np.mean(tempos_ms)),
        "p95_ms": float(np.percentile(tempos_ms, 95)),
        "min_ms": float(np.min(tempos_ms)),
        "max_ms": float(np.max(tempos_ms)),
        "input_dtype": str(input_detail["dtype"]),
        "output_dtype": str(output_detail["dtype"]),
        "input_quantization": [float(x) for x in input_detail["quantization"]],
        "output_quantization": [float(x) for x in output_detail["quantization"]],
    }


def listar_operadores_tflite(tflite_path):
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))

    interpreter.allocate_tensors()

    ops = interpreter._get_ops_details()

    return sorted({op["op_name"] for op in ops})


def escrever_header_c(tflite_path, header_path, array_name):
    data = Path(tflite_path).read_bytes()

    with open(header_path, "w", encoding="utf-8") as f:
        f.write("#pragma once\n\n")
        f.write("#include <cstdint>\n\n")
        f.write(f"alignas(16) const unsigned char {array_name}[] = {{\n")

        for i in range(0, len(data), 12):
            chunk = data[i : i + 12]

            linha = ", ".join(f"0x{byte:02x}" for byte in chunk)

            f.write(f"  {linha},\n")

        f.write("};\n")
        f.write(f"const unsigned int {array_name}_len = {len(data)};\n")


def salvar_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


def run_tflite_optimization(model_info):
    name = model_info["name"]
    model_path = model_info["model_path"]
    dataset_path = model_info["dataset_path"]
    output_base_dir = model_info["output_base_dir"]
    step_seconds = model_info["step_seconds"]
    representative_samples = model_info["representative_samples"]
    batch_size = model_info["batch_size"]
    benchmark_limit = model_info["benchmark_limit"]


    out_dir = output_base_dir / name
    out_dir.mkdir(parents=True, exist_ok=True)

    dataset = carregar_dataset(dataset_path)

    model = tf.keras.models.load_model(model_path)

    input_shape = tuple(model.input_shape)

    resultado = {
        "modelo": name,
        "keras_model_path": str(model_path),
        "dataset": str(dataset_path),
        "input_shape": str(input_shape),
        "params": int(model.count_params()),
        "step_seconds": step_seconds,
        "representative_samples": representative_samples,
        "avaliacoes": {},
    }

    print("=" * 80)
    print(f"AVALIANDO KERAS ORIGINAL ({name})")
    print("=" * 80)

    scores_val_keras = predizer_keras(
        model,
        dataset["X_val"],
        input_shape,
        batch_size,
    )

    scores_test_keras = predizer_keras(
        model,
        dataset["X_test"],
        input_shape,
        batch_size,
    )

    metricas_keras = avaliar_com_validacao(
        dataset["y_val"],
        scores_val_keras,
        dataset["y_test"],
        scores_test_keras,
        step_seconds,
    )

    np.save(out_dir / "scores_val_keras.npy", scores_val_keras)
    np.save(out_dir / "scores_test_keras.npy", scores_test_keras)

    resultado["avaliacoes"]["keras"] = {
        "metricas": metricas_keras,
    }

    conversores = {
        "float32": lambda: converter_float32(model),
        "dynamic_range": lambda: converter_dynamic_range(model),
        "float16": lambda: converter_float16(model),
        "int8": lambda: converter_int8(
            model,
            dataset["X_train"],
            input_shape,
            representative_samples,
        ),
    }

    for conv_name, func_converter in conversores.items():
        print()
        print("=" * 80)
        print(f"CONVERTENDO E AVALIANDO: {conv_name} ({name})")
        print("=" * 80)

        try:
            inicio = time.time()

            modelo_tflite = func_converter()

            tempo_conversao = time.time() - inicio

            tflite_path = out_dir / f"{name}_{conv_name}.tflite"

            tflite_path.write_bytes(modelo_tflite)

            header_path = out_dir / f"{name}_{conv_name}.h"

            escrever_header_c(
                tflite_path,
                header_path,
                array_name=f"g_{name.replace('-', '_')}_{conv_name}_model", # Dynamic array name
            )

            scores_val, bench_val = predizer_tflite(
                tflite_path,
                dataset["X_val"],
                input_shape,
            )

            scores_test, bench_test = predizer_tflite(
                tflite_path,
                dataset["X_test"],
                input_shape,
            )

            _, bench_curto = predizer_tflite(
                tflite_path,
                dataset["X_test"],
                input_shape,
                limite=benchmark_limit,
            )

            metricas = avaliar_com_validacao(
                dataset["y_val"],
                scores_val,
                dataset["y_test"],
                scores_test,
                step_seconds,
            )

            np.save(out_dir / f"scores_val_{conv_name}.npy", scores_val)
            np.save(out_dir / f"scores_test_{conv_name}.npy", scores_test)

            operadores = listar_operadores_tflite(tflite_path)

            resultado["avaliacoes"][conv_name] = {
                "ok": True,
                "tflite_path": str(tflite_path),
                "header_path": str(header_path),
                "size_kb": float(tflite_path.stat().st_size / 1024),
                "tempo_conversao_s": float(tempo_conversao),
                "operadores": operadores,
                "benchmark_val": bench_val,
                "benchmark_test": bench_test,
                "benchmark_curto": bench_curto,
                "metricas": metricas,
            }

            teste = metricas["teste"]

            print(
                f"{conv_name}: "
                f"size_kb={resultado['avaliacoes'][conv_name]['size_kb']:.3f} | "
                f"AUC-PR={teste['auc_pr']:.4f} | "
                f"F1={teste['f1']:.4f} | "
                f"ms={bench_curto['mean_ms']:.4f}"
            )

        except Exception as exc:
            resultado["avaliacoes"][conv_name] = {
                "ok": False,
                "erro": repr(exc),
            }

            print(f"Falhou {conv_name}: {exc}")

    salvar_json(resultado, out_dir / "quantizacao_avaliacao_result.json")

    print()
    print("=" * 80)
    print(f"RESUMO FINAL ({name})")
    print("=" * 80)

    for conv_name, info in resultado["avaliacoes"].items():
        if conv_name == "keras":
            teste = info["metricas"]["teste"]
            print(
                f"{conv_name}: "
                f"AUC-PR={teste['auc_pr']:.4f} | "
                f"F1={teste['f1']:.4f}"
            )
            continue

        if not info["ok"]:
            print(f"{conv_name}: FALHOU")
            continue

        teste = info["metricas"]["teste"]

        print(
            f"{conv_name}: "
            f"{info['size_kb']:.2f} KB | "
            f"AUC-PR={teste['auc_pr']:.4f} | "
            f"F1={teste['f1']:.4f} | "
            f"mean_ms={info['benchmark_curto']['mean_ms']:.4f}"
        )

    print("Salvo em:", out_dir / "quantizacao_avaliacao_result.json")


if __name__ == "__main__":
    # Define parameters for each model
    common_args = {
        "dataset_path": DATASET_V4,
        "output_base_dir": RESULTS_DIR / "tflite_optimization",
        "step_seconds": STEP_SECONDS,
        "representative_samples": 500,
        "batch_size": 2048,
        "benchmark_limit": 300,
    }

    model_configs = [
        {
            "name": "optuna_tiny_cnn_classifier",
            "model_path": RESULTS_DIR / "optuna_tiny_cnn_classifier" / "optuna_tiny_cnn_classifier.keras",
            **common_args,
        },
        {
            "name": "optuna_tiny_tcn_classifier",
            "model_path": RESULTS_DIR / "optuna_tiny_tcn_classifier" / "optuna_tiny_tcn_classifier.keras",
            **common_args,
        },
    ]

    for config in model_configs:
        run_tflite_optimization(config)


AVALIANDO KERAS ORIGINAL (optuna_tiny_cnn_classifier)

CONVERTENDO E AVALIANDO: float32 (optuna_tiny_cnn_classifier)
Saved artifact at '/tmp/tmpoujlugq6'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 800, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138582377318160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377318544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377317968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377320080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377322576: TensorSpec(shape=(), dtype

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


float32: size_kb=66.055 | AUC-PR=0.9139 | F1=0.8532 | ms=0.0995

CONVERTENDO E AVALIANDO: dynamic_range (optuna_tiny_cnn_classifier)
Saved artifact at '/tmp/tmprbxw21c9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 800, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138582377318160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377318544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377317968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377320080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377322576: TensorSpec

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


dynamic_range: size_kb=24.156 | AUC-PR=0.9140 | F1=0.8532 | ms=0.1171

CONVERTENDO E AVALIANDO: float16 (optuna_tiny_cnn_classifier)
Saved artifact at '/tmp/tmpx055uslx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 800, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138582377318160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319120: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377318544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377319504: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377317968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377320080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582377322576: TensorSpec

...

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


float16: size_kb=49.699 | AUC-PR=0.9273 | F1=0.8703 | ms=0.5196

CONVERTENDO E AVALIANDO: int8 (optuna_tiny_tcn_classifier)
Saved artifact at '/tmp/tmp15ba2ov5'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 800, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  138582372977744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372976208: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372985616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372984848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372976016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372983312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372975824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372983696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138582372982736: TensorSpec(shape=()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


int8: size_kb=36.430 | AUC-PR=0.9175 | F1=0.8633 | ms=0.7725

RESUMO FINAL (optuna_tiny_tcn_classifier)
keras: AUC-PR=0.9273 | F1=0.8703
float32: 82.41 KB | AUC-PR=0.9273 | F1=0.8703 | mean_ms=0.5146
dynamic_range: 37.77 KB | AUC-PR=0.9271 | F1=0.8703 | mean_ms=0.6202
float16: 49.70 KB | AUC-PR=0.9273 | F1=0.8703 | mean_ms=0.5196
int8: 36.43 KB | AUC-PR=0.9175 | F1=0.8633 | mean_ms=0.7725
Salvo em: /content/drive/MyDrive/AlvaroSampaio/TCC/TCC_data/resultados/tflite_optimization/optuna_tiny_tcn_classifier/quantizacao_avaliacao_result.json


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
